07/02/2024: 

As an alternative to the big mixed-effects Bayesian GLM, let's use this space to run electrode-level regressions with permutation testing. 

09/25/2024: 

My general strategy here is to: 


1. Compute an electrode GLM to test RPE sensitivity, RPE:phit interaction during encoding, and memory sensitivity during the cue period. These electrode-level effects are then aggregated within region and region-level effects are assessed using a mixed-effects model to see if there are consistent effects across patients. 

In [ ]:
# %matplotlib inline
# %reload_ext autoreload
# %autoreload 2

In [ ]:
import arviz as az
import bambi as bmb
import argparse
from os.path import join
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt 
from scipy.stats import zscore, spearmanr, pearsonr, ttest_1samp
import os
from statsmodels.api import OLS
from scipy import stats
from tqdm import tqdm
import patsy
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import seaborn as sns
import xarray as xr
from matplotlib.backends.backend_pdf import PdfPages


In [ ]:
import sys
sys.path.append('/hpc/users/qasims01/resources/LFPAnalysis')
from LFPAnalysis import statistics_utils

## set some plotting parameters

In [ ]:
# Set some plot defaults for matplotlib (when not using seaborn): 

import  matplotlib.font_manager

path = '/sc/arion/projects/guLab/Salman/Arial.ttf'
matplotlib.font_manager.fontManager.addfont(path)

prop = matplotlib.font_manager.FontProperties(fname=path)

# Set the default font family to the registered custom font
plt.rcParams['font.family'] = prop.get_name()

# Rest of your customization code
plt.rcParams['font.size'] = 20
plt.rcParams['lines.linewidth'] = 1

# Set palette: 

# Define RGB values for your two colors
ignacio_purple = np.array([86, 28, 70]) / 255
ignacio_blue = np.array([0, 70, 126]) / 255 

# Create a Seaborn palette with your custom colors normalized to 0-1
custom_palette = sns.color_palette([ignacio_purple, ignacio_blue])

ignacio_orange = np.array([255, 172, 77]) / 255
ignacio_mauve = np.array([193, 155, 211]) / 255
ignacio_beige = np.array([205, 182, 162]) / 255

# Set the custom palette
sns.set_palette(custom_palette)
sns.set_context("paper")

# from YBA
hpc_c = np.array([0.44314, 0.2902, 1])
amy_c = np.array([0.49412, 0, 0.90196])
ofc_c = np.array([1, 0.75686, 0.41569])
acc_c = np.array([0.7098, 0.27059 ,0.68235])
dmpfc_c = np.array([0.40784, 0.01961, 0])
dlpfc_c = np.array([1, 0.06275, 0.06275])

## Load the behavioral data: 

In [ ]:
base_dir = '/sc/arion' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
save_dir_plots = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Plots'


# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site!='UI']

# Let's get rid of the patients who do worse than chance on the DM task, or have no hits in the memory task

subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

In [ ]:
import statsmodels.api as sm

def multinomial_regression(df, response_col, predictor_col):
    # Drop rows with NaN values in the dependent or predictor columns
    rel_cols = [response_col] + predictor_col
    df_clean = df[rel_cols].dropna()

    # Encode the categorical response variable
    # label_encoder = LabelEncoder()
    # df_clean['response_encoded'] = label_encoder.fit_transform(df_clean[response_col].values.ravel())
    df_clean['response_encoded'] = df_clean[response_col].apply(lambda x: 0 if x=='hits' else 1 if x=='misses' else 2 if x=='false_alarms' else 3).values.ravel()
    
    # Prepare the predictor variable (add constant for intercept)
    X = sm.add_constant(df_clean[predictor_col])
    
    # The dependent variable is the encoded response
    y = df_clean['response_encoded']
        
    # Fit the multinomial logistic regression model
    model = sm.MNLogit(y, X)
    result = model.fit(disp=0)
    
    # Return the model result and the label encoder for further use
    return result

# Example usage
# result = multinomial_regression(combined_df, 'memory_outcome', ['recog_rt_log'])
# any(result.pvalues.loc['recog_rt_log'].values <=0.05)

Pre-process the EMU behavioral data

In [ ]:
learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

combined_df['gamble_rt_log'] = np.log(combined_df.gamble_rt_gamble)
combined_df['recog_rt_log'] = np.log(combined_df.recog_rt)


In [ ]:
combined_df['memory_outcome'] = combined_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome') 

In [ ]:
learn_df = learn_df[learn_df.gamble_rt>0.3]
learn_df = learn_df[learn_df.gamble_rt<10]

drop_trials = pd.concat([combined_df[combined_df.gamble_rt_gamble<0.3], 
                          combined_df[combined_df.gamble_rt_gamble>10], 
                          combined_df[combined_df.recog_rt<0.3], 
                          combined_df[combined_df.recog_rt>10]])

combined_df.drop(index=drop_trials.index, inplace=True)

# Cue analysis: 

## predict the upcoming memory response using HFA

In [ ]:
bands = ['high_theta']
rois =['dlPFC', 'OFC', 'dmPFC', 'AMY', 'HPC'] 
analysis_evs = 'cue_on'
block = [0, 0.5]
run_surr = True
nsurr = 500
memory_responses = ['hits', 'misses', 'false_alarms', 'correct_rejections'] 

result_dfs = [] 

for band in bands: 
    
    chan_dfs = [] 
    glms = []
    elec_counter = 0 
    for subj_id in subj_ids:
        print(f'Loading subj {subj_id}')
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
        for chan in elec_df.label.unique():
            if 'lesion' in chan:
                # skip the lesion channels (dmPFC)
                print('skipping lesion channel')
                continue
                
            if elec_df[elec_df.label==chan].salman_region.values[0] in rois:
                chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time{block[0]}-{block[1]}.csv')
                chan_df['unique_elec_id'] = elec_counter
                
                # Run permutation regression
                chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
                chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
                chan_df['recog_rt_log'] = np.log(chan_df.recog_rt)
                
                rpe_results = statistics_utils.permutation_regression_zscore(chan_df[chan_df.fband==band],
                                                                         'tfr ~ 1 + zrpe + zpm + recog_rt_log', n_permutations=1000, plot_res=False)
                
                chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome') 
                
                # chan_df.loc[chan_df['memory_outcome'].isin(['misses', 'false_alarms']), 'memory_outcome'] = 'bad'
                # chan_df.loc[chan_df['memory_outcome'].isin(['hits', 'correct_rejections']), 'memory_outcome'] = 'good'
                
                rel_col = ['tfr', 'memory_outcome', 'recog_rt_log']
                
                model_df =  chan_df[(chan_df.fband==band)].reset_index(drop=True)[rel_col].dropna()
                
                result = multinomial_regression(model_df, 'memory_outcome', ['tfr', 'recog_rt_log'])
                
                result_df = pd.DataFrame(columns=['participant', 'band', 'label', 'region', 
                                                  'coefficient', 'coefficient_z', 'prsquared', 'prsquared_z', 'category'])
                
                result_df['participant'] = [subj_id]*3
                result_df['band'] = [band]*3
                result_df['label'] = [chan]*3
                result_df['region'] = [elec_df[elec_df.label==chan].salman_region.values[0]]*3
                result_df['prsquared'] = result.prsquared
                result_df['rpe'] = rpe_results.loc['zrpe']['Z_Score']
                result_df['pm'] = rpe_results.loc['zpm']['Z_Score']
                result_df.loc[0, 'coefficient'] = result.params.loc['tfr'].values[0]
                result_df.loc[0, 'category'] = 'miss'
                result_df.loc[1, 'coefficient'] = result.params.loc['tfr'].values[1]
                result_df.loc[1, 'category'] = 'false_alarm'
                result_df.loc[2, 'coefficient'] = result.params.loc['tfr'].values[2]
                result_df.loc[2, 'category'] = 'correct_rejection'
                
                if run_surr: 
                    surr_params = np.zeros([3, nsurr]) 
                    surr_rsquared = [] 
                    for surr in range(nsurr): 
                        surr_df = model_df.copy()
                        surr_df['memory_outcome'] = surr_df['memory_outcome'].sample(frac=1).values
                        try:
                            surr_res = multinomial_regression(surr_df, 'memory_outcome', ['tfr', 'recog_rt_log'])
                        except: 
                            continue
                        surr_param = surr_res.params.loc['tfr'].values
                        surr_params[:, surr] = surr_param
                        surr_rsquared.append(surr_res.prsquared)

                    real_param = result.params.loc['tfr'].values

                    real_param_z = (real_param - np.nanmean(surr_params, axis=1)) / np.nanstd(surr_params, axis=1)
                    
                    rsquared_z = (result.prsquared - np.nanmean(surr_rsquared)) / np.nanstd(surr_rsquared)
                    result_df['prsquared_z'] = rsquared_z
                    
                    # Collect all z-scored coefficients: 
                    result_df.loc[0, 'coefficient_z'] = real_param_z[0]
                    result_df.loc[1, 'coefficient_z'] = real_param_z[1]
                    result_df.loc[2, 'coefficient_z'] = real_param_z[2]

                result_dfs.append(result_df)
                elec_counter +=1

In [ ]:
full_data_df = pd.concat(result_dfs)

In [ ]:
full_data_df['hit_compared_to_coefficient'] = -full_data_df.coefficient_z

In [ ]:
# save out full_data
full_data_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_multinomial_df_wrpe', index=False) 


In [ ]:
full_data_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_multinomial_df')
all_elec_dfs = []
# subj_ids = ['UI001', 'UI002', 'UI003', 'UI004']
for subj_id in subj_ids: 
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
    elec_df['participant'] = subj_id
    all_elec_dfs.append(elec_df)
all_elec_dfs = pd.concat(all_elec_dfs).reset_index() 

In [ ]:
merged_elec_df = all_elec_dfs.merge(full_data_df[full_data_df.region=='OFC'], on=['participant', 'label'])

In [ ]:
merged_elec_df[['mni_x', 'mni_y', 'mni_z', 'hit_compared_to_coefficient']].to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_hfa_OFC_elecs', index=False)


In [ ]:
# Create the figure and axis
fig, ax = plt.subplots(figsize=(9, 5))
    
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_spatial_location_memory.pdf') as pdf:

    merged_elec_df['mni_z_bin'] = pd.cut(merged_elec_df['mni_z'], bins=5)
    merged_elec_df['mni_y_bin'] = pd.cut(merged_elec_df['mni_y'], bins=5)

    # Group by the binned coordinates and calculate the mean of 'phit' for each bin
    df_binned = merged_elec_df.groupby(['mni_z_bin', 'mni_y_bin'])['hit_compared_to_coefficient'].mean().reset_index()

    # Pivot the dataframe for the heatmap
    heatmap_data = df_binned.pivot(columns="mni_y_bin", index="mni_z_bin", values="hit_compared_to_coefficient")

    # Plot the heatmap on the provided axis
    sns.heatmap(heatmap_data, cmap="viridis", annot=False, ax=ax)

    # Customize the plot
    ax.set_title("Heatmap of Phit by Binned MNI coordinates")
    plt.xticks(rotation=45)

    ax.invert_yaxis()
    ax.invert_xaxis()
    # Show the plot
    plt.show()
    pdf.savefig(fig)


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=300)
# with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_rpe_eleclevel.pdf') as pdf:
sns.barplot(data=full_data_df[(full_data_df.category=='miss')], x='region', y='rpe', errorbar='se', legend=False, hue='category',
        order = ['dmPFC',  'dlPFC', 'OFC', 'HPC', 'AMY'], color= 'k', alpha = 0.5)
sns.stripplot(data=full_data_df[(full_data_df.category=='miss') & (full_data_df['rpe']<=1.96)], x='region', y='rpe', legend=False,
    order = ['dmPFC',  'dlPFC', 'OFC',  'HPC', 'AMY'], alpha=0.1, color='k')
sns.stripplot(data=full_data_df[(full_data_df.category=='miss') & (full_data_df['rpe']>1.96)], x='region', y='rpe',legend=False,
    order = ['dmPFC',  'dlPFC', 'OFC', 'HPC', 'AMY'], color='r',  alpha=0.5)

sns.despine()
plt.hlines([0], -1, 6, 'k')
ax.set(ylabel='Hit vs. Miss sensitivity', xlim=[-1,6], ylim=[-3.75, 3.75])
    # pdf.savefig(fig)


There is no significant rpe coding during the cue presentation!

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=300)
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_memory_eleclevel.pdf') as pdf:
    sns.barplot(data=full_data_df[(full_data_df.category=='miss')], x='region', y='hit_compared_to_coefficient', errorbar='se', legend=False, hue='category',
            order = ['dmPFC',  'dlPFC', 'OFC', 'HPC', 'AMY'], color= 'k', alpha = 0.5)
    sns.stripplot(data=full_data_df[(full_data_df.category=='miss') & (full_data_df['hit_compared_to_coefficient']<=1.96)], x='region', y='hit_compared_to_coefficient', legend=False,
        order = ['dmPFC',  'dlPFC', 'OFC',  'HPC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=full_data_df[(full_data_df.category=='miss') & (full_data_df['hit_compared_to_coefficient']>1.96)], x='region', y='hit_compared_to_coefficient',legend=False,
        order = ['dmPFC',  'dlPFC', 'OFC', 'HPC', 'AMY'], color='r',  alpha=0.5)

    sns.despine()
    plt.hlines([0], -1, 6, 'k')
    ax.set(ylabel='Hit vs. Miss sensitivity', xlim=[-1,6], ylim=[-3.75, 3.75])
    pdf.savefig(fig)


Perform the mixed-effects model test to see if this effect is consistent across patients

In [ ]:
region = 'OFC'
predictor = 'hit_compared_to_coefficient'
model_df = full_data_df[(full_data_df.region==region) & (full_data_df.category=='miss')]

model_df[f'{predictor}'] = model_df[f'{predictor}'].astype(float)

model = bmb.Model(f'{predictor} ~ 1 + (1|participant)', 
                  model_df, 
                  family='gaussian', categorical=['participant'])

# Fit the model
results = model.fit(draws=1000, chains=4, target_accept=0.9)

df = az.summary(results)

# if results.pvalues['Intercept']<=0.025:
if np.sign(df.loc['Intercept']['hdi_3%']) == np.sign(df.loc['Intercept']['hdi_97%']):
    
    # print(results.summary())
    # with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_{region}_{predictor}_M_on_betas.pdf') as pdf:

    # Create a scatter plot of individual data points
    fig, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
    sns.stripplot(x="participant", y=f'{predictor}', data=model_df, 
                  jitter=True, alpha=0.6, color="gray", s=6, ax=ax)

    # plot the mean line
    subject_means = model_df.groupby("participant")[f'{predictor}'].mean()

    sns.boxplot(showmeans=True,
                meanline=True,
                meanprops={'color': 'k', 'ls': '-', 'lw': 2},
                medianprops={'visible': False},
                whiskerprops={'visible': False},
                zorder=10,
                x=subject_means.index,
                y=subject_means.values,
                showfliers=False,
                showbox=False,
                showcaps=False,
                ax=ax)


    # # Overlay the subject means

    # plt.scatter(subject_means.index, subject_means.values, color="red", label="Subject Means", zorder=3)

    # Plot the overall mean from the mixed-effects model
    # overall_mean = results.fe_params['Intercept']
    plt.axhline(df.loc['Intercept']['mean'], color="blue", linestyle="--", label=f"Overall Mean: {df.loc['Intercept']['mean']:.2f}")
    plt.axhline(df.loc['Intercept']['hdi_3%'], color='red', linestyle='--')
    plt.axhline(df.loc['Intercept']['hdi_97%'], color='red', linestyle='--')

    plt.axhline(0, color='black')
    sns.despine()
        
        # pdf.savefig()

In [ ]:
df

In [ ]:
# prop_df = 100 * full_data_df[full_data_df.abs_value>1.96][['region', 'category']].value_counts() / (full_data_df[['region']].value_counts()/3)
# prop_df = pd.DataFrame(prop_df).reset_index()




In [ ]:
# sub_region_df = pd.DataFrame(full_data_df[full_data_df.abs_value>1.96][['participant', 'region', 'category']].value_counts()).reset_index()



9/21/2024: 

My tactic for the retrieval stage is to determine whether HFA in a region can distinguish a hit from at least one of the following: [miss, false alarm, correct rejection]. If so, I consider the electrode "hit sensitive". 

This is mostly dmPFC and OFC!

This is important because I want to make the argument that hit sensitivity is modulated by RPE. 

## Let's see what the theta coherence looks like during cue

In [ ]:
metric = 'coh'
band1 = 'theta'
band2 = 'theta'
event = 'cue_on' 
savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

pair = ['dlPFC', 'OFC']

all_pairs_df = pd.read_csv(f'{savepath}/{pair[0]}_{pair[1]}_{metric}_{band1}_{band2}_{event}_df')


In [ ]:
all_pairs_df['label'] = np.nan
all_pairs_df['label'] = all_pairs_df.pair_label.apply(lambda x: x.split(':')[1])

In [ ]:
# all_pairs_df.reset_index(drop=True, inplace=True)
# all_pairs_df['label'] = all_pairs_df.pair_label.apply(lambda x: x.split(':')[1])
# # full_data_df = full_data_df.groupby(['participant', 'label', 'roi1']).mean(numeric_only=True).reset_index()
                                    
# all_pairs_df = all_pairs_df.loc[all_pairs_df.groupby(['participant', 'label', 'roi1'])['conn'].idxmax()]
# #.idxmax()]
# #.apply(lambda x: x.abs().idxmax())]


In [ ]:
all_pairs_df

In [ ]:
full_data_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_multinomial_df') 


In [ ]:
grouped_cue_df = full_data_df[(full_data_df.region=='OFC') & (full_data_df.category=='miss')].groupby(['participant', 'label']).mean(numeric_only=True).reset_index()


In [ ]:
cue_merge = grouped_cue_df.merge(all_pairs_df.groupby(['participant', 'label']).mean(numeric_only=True).reset_index(),
                                 on=['participant', 'label'])


In [ ]:
    sns.lmplot(data=cue_merge, x='conn', y='hit_compared_to_coefficient', hue='participant', ci=None)
    sns.regplot(data=cue_merge, x='conn', y='hit_compared_to_coefficient', color='k', ci=95, scatter=None)
    

In [ ]:
results = smf.mixedlm('hit_compared_to_coefficient ~ 1 + conn', data=cue_merge, 
                      re_formula = '1 + conn', 
                      groups='participant').fit(reml=False)

In [ ]:
results.summary()

# Encoding analysis:

In [ ]:
rois = ['dmPFC', 'OFC', 'HPC', 'AMY', 'ACC', 'dlPFC']

analysis_evs = 'feedback_start'

## Control stuff looking at reward coding:

In [ ]:
# band = ['theta', 'alpha', 'beta', 'hfa'] 


# # iterate through subjects

# model_results = {'participant':[], 'channel':[], 'region':[], 'unique_elec_id':[], 
#                  'delta_bic':[], 'band':[]} 

# for subj_id in subj_ids:

#     print(f'Loading subj {subj_id}')
#     day = 'Day1'
#     filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
#     elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
    
#     # iterate through channels 
#     elec_counter = 0 
#     for chan in elec_df.label.unique():
#         if 'lesion' in chan:
#             # skip the lesion channels (dmPFC)
#             print('skipping lesion channel')
#             continue
#         if elec_df[elec_df.label==chan].salman_region.values[0] in rois:
#             chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time0-1.5.csv') # _time{block[0]}-{block[1]}
#             chan_df['unique_elec_id'] = elec_counter
#             chan_df = chan_df[['participant', 'region', 'channel', 'unique_elec_id', 'trials', 'fband', 'tfr']]

#             chan_df = chan_df.merge(learn_df, on=['participant', 'trials'])

#             chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
#             chan_df['binarized_rpe'] = chan_df.rpe.apply(lambda x: 1 if x>0 else -1)
#             chan_df['urpe'] = np.abs(chan_df.rpe)
#             chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
#             chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
#             chan_df['zreward'] = chan_df.reward.apply(lambda x: -1 if x==0 else x) 
                
#             # iterate through bands 
#             for band in bands: 

#                 # compute reward regression
#                 formula = 'tfr ~ 1 + zreward'
#                 rel_col = ['tfr', 'zreward']
#                 model_df =  chan_df[(chan_df.fband==band)].reset_index(drop=True)[rel_col].dropna()
#                 reward_res = smf.ols(formula=formula, data=model_df).fit()

#                 # add rpe to the model
#                 formula = 'tfr ~ 1 + zrpe + zreward'
#                 rel_col = ['tfr', 'zrpe', 'zreward']
#                 model_df =  chan_df[(chan_df.fband==band)].reset_index(drop=True)[rel_col].dropna()
#                 rpe_res = smf.ols(formula=formula, data=model_df).fit()

#                 # compute bic improvement
#                 delta_bic = rpe_res.rsquared - reward_res.rsquared

#                 model_results['participant'].append(chan_df.participant.unique()[0])
#                 model_results['channel'].append(chan_df.channel.unique()[0])
#                 model_results['region'].append(chan_df.region.unique()[0])
#                 model_results['unique_elec_id'].append(chan_df.unique_elec_id.unique()[0])
#                 model_results['delta_bic'].append(delta_bic)
#                 model_results['band'].append(band)

# rpe_bic_df = pd.DataFrame(model_results)


In [ ]:
# for band in ['theta', 'hfa']:
#     print(band)
#     print(rpe_bic_df[(rpe_bic_df.band==band) & (rpe_bic_df.delta_bic<=-2)].region.value_counts() / rpe_bic_df[rpe_bic_df.band==band].region.value_counts())



What biomarkers show improved BIC for RPE after accounting for reward? 

Theta: OFC 

HFA: OFC 

Honorable mention: Beta OFC 

What about looking at just the degree of r-squared improvement by band by region?

In [ ]:
# Control: reward, instead of rpe

In [ ]:
# bands = ['hfa']
# # blocks = [[0, 0.5], [0.5, 1], [1, 1.5]]

# for band in bands: 
    
#     r2_results = {'rpe':[], 'reward':[], 'urpe':[]} 
    
#     # for block in blocks:
#     # RELOAD the data each time the model is run to ensure no funny business with dataframe manipulation between loop runs
#     chan_dfs = [] 
#     glms = []
#     elec_counter = 0 
#     for subj_id in subj_ids:
#         print(f'Loading subj {subj_id}')
#         day = 'Day1'
#         filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
#         elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

#         for chan in elec_df.label.unique():
#             if 'lesion' in chan:
#                 # skip the lesion channels (dmPFC)
#                 print('skipping lesion channel')
#                 continue
#             if elec_df[elec_df.label==chan].salman_region.values[0] in rois:
#                 chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time0-1.5.csv') # _time{block[0]}-{block[1]}
#                 chan_df['unique_elec_id'] = elec_counter
#                 chan_df = chan_df[['participant', 'region', 'channel', 'unique_elec_id', 'trials', 'fband', 'tfr']]

#                 chan_df = chan_df.merge(learn_df, on=['participant', 'trials'])

#                 chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
#                 chan_df['binarized_rpe'] = chan_df.rpe.apply(lambda x: 1 if x>0 else -1)
#                 chan_df['urpe'] = np.abs(chan_df.rpe)
#                 chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
#                 chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
#                 chan_df['zreward'] = chan_df.reward.apply(lambda x: -1 if x==0 else x) 
         
#                 # try the different alternatives: 
                
#                 formula = 'tfr ~ 1 + zreward'
#                 rel_col = ['tfr', 'zreward']
#                 model_df =  chan_df[(chan_df.fband==band)].reset_index(drop=True)[rel_col].dropna()
#                 model_res = smf.ols(formula=formula, data=model_df).fit()
#                 r2_results['reward'].append(model_res.rsquared)
                
#                 formula = 'tfr ~ 1 + zrpe + zreward'
#                 rel_col = ['tfr', 'zrpe', 'zreward']
#                 model_df =  chan_df[(chan_df.fband==band)].reset_index(drop=True)[rel_col].dropna()
#                 model_res = smf.ols(formula=formula, data=model_df).fit()
#                 r2_results['rpe'].append(model_res.rsquared)


In [ ]:
# # Create a DataFrame from the dictionary
# df = pd.DataFrame.from_dict(r2_results, orient='columns')

# # Convert the DataFrame into a long format
# df_long = df.stack().reset_index()

# # Rename the columns
# df_long.columns = ['index', 'key', 'value']

# # Drop the unnecessary index column if needed
# df_long = df_long.drop(columns='index')

In [ ]:
# sns.barplot(data=df_long, x='key', y='value')

In [ ]:
# def partial_correlation(x, y, z):
#     """
#     Compute partial correlation between x and y, controlling for z.
    
#     Parameters:
#     x, y, z : 1D numpy arrays or pandas Series
    
#     Returns:
#     partial correlation coefficient
#     """
#     x_resid = sm.OLS(x, sm.add_constant(z)).fit().resid
#     y_resid = sm.OLS(y, sm.add_constant(z)).fit().resid
    
#     return np.corrcoef(x_resid, y_resid)[0, 1]

In [ ]:

# rois = ['dmPFC', 'OFC', 'HPC', 'AMY', 'ACC', 'dlPFC']
# bands = ['theta', 'alpha', 'beta', 'hfa']

# for band in bands: 
        
#     model_results = {'participant':[], 'channel':[], 'region':[], 'unique_elec_id':[], 
#                  'rpe':[], 'pm':[], 'phit':[]} 
    
#     # for block in blocks:
#     # RELOAD the data each time the model is run to ensure no funny business with dataframe manipulation between loop runs
#     chan_dfs = [] 
#     glms = []
#     elec_counter = 0 
#     for subj_id in subj_ids:
#         print(f'Loading subj {subj_id}')
#         day = 'Day1'
#         filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
#         elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

#         for chan in elec_df.label.unique():
#             if 'lesion' in chan:
#                 # skip the lesion channels (dmPFC)
#                 print('skipping lesion channel')
#                 continue
#             if elec_df[elec_df.label==chan].salman_region.values[0] in rois:
#                 chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time0-1.5.csv') # _time{block[0]}-{block[1]}
#                 chan_df['unique_elec_id'] = elec_counter
#                 chan_df = chan_df[['participant', 'region', 'channel', 'unique_elec_id', 'trials', 'fband', 'tfr']]

#                 chan_df = chan_df.merge(learn_df, on=['participant', 'trials'])

#                 chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
#                 chan_df['binarized_rpe'] = chan_df.rpe.apply(lambda x: 1 if x>0 else -1)
#                 chan_df['urpe'] = np.abs(chan_df.rpe)
#                 chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
#                 chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
                                        
#                 model_results['participant'].append(chan_df.participant.unique()[0])
#                 model_results['channel'].append(chan_df.channel.unique()[0])
#                 model_results['region'].append(chan_df.region.unique()[0])
#                 model_results['unique_elec_id'].append(chan_df.unique_elec_id.unique()[0])

#                 formula = 'tfr ~ 1 + zrpe + zpm'
                
#                 rel_col = ['tfr', 'zrpe', 'zpm']
#                 model_df =  chan_df[(chan_df.fband==band)].reset_index(drop=True)[rel_col].dropna()

#                 results = permutation_regression_zscore(model_df, 
#                                                         formula, 
#                                                         n_permutations=1000, 
#                                                         plot_res=False)

#                 model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
#                 model_results['pm'].append(results.loc['zpm']['Z_Score'])

#                 memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
#                 memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
#                 memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
#                 chan_df = chan_df.merge(memory_df, on=['participant', 'trials'])
                
#                 formula = 'tfr ~ 1 + C(phit)'
                
#                 rel_col = ['tfr', 'phit']
#                 model_df =  chan_df[(chan_df.fband==band)].reset_index(drop=True)[rel_col].dropna()

#                 results = permutation_regression_zscore(model_df, 
#                                                         formula, 
#                                                         n_permutations=1000, 
#                                                         plot_res=False)

#                 model_results['phit'].append(results.loc['C(phit)[T.1]']['Z_Score'])

#                 elec_counter +=1

#     gamble_model_df = pd.DataFrame(model_results)
#     gamble_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_rpe_pm_phit_no_intx_elec_level_df', index=False)


## Main POWER analysis of RPE sensitivity across frequencies:

In [ ]:
rois = ['AMY', 'HPC', 'dmPFC', 'dlPFC', 'OFC']
bands = ['low_theta', 'high_theta']
analysis_evs = 'feedback_start'

for band in bands: 
    model_results = {'participant':[], 'channel':[], 'region':[], 'unique_elec_id':[], 'rpe':[], 'pm':[], 'phit':[], 'phit:zrpe':[]} 
    chan_dfs = [] 
    elec_counter = 0 
    
    for subj_id in subj_ids:
        
        print(f'Loading subj {subj_id}')
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        # Pre-filter for ROIs and to omit lesion electrodes
        valid_elec_df = elec_df[elec_df.salman_region.isin(rois) & ~elec_df.label.str.contains('lesion')]

        for chan in valid_elec_df.label.unique():
            
            chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time0-1.5.csv')
            # chan_df = chan_df[['region', 'channel', 'fband', 'trial', 'tfr']]
            # chan_df.rename(columns={'trial':'trials'}, inplace=True) 
            chan_df['participant'] = subj_id
            
            chan_df['unique_elec_id'] = elec_counter
            chan_df = chan_df[['participant', 'region', 'channel', 'unique_elec_id', 'trials', 'fband', 'tfr']]

            chan_df = chan_df.merge(learn_df, on=['participant', 'trials'])

            zrpe_mean = np.nanmean(chan_df.rpe)
            zrpe_std = np.nanstd(chan_df.rpe)
            chan_df['zrpe'] = (chan_df.rpe - zrpe_mean) / (2 * zrpe_std)
            chan_df['urpe'] = np.abs(chan_df.rpe)

            zpm_mean = np.nanmean(chan_df.DPRIME)
            zpm_std = np.nanstd(chan_df.DPRIME)
            chan_df['zpm'] = (chan_df.DPRIME - zpm_mean) / (2 * zpm_std)

            chan_df_band = chan_df[chan_df.fband == band]
            chan_df_band = chan_df_band.dropna(subset=['tfr', 'zrpe', 'zpm'])

            # Run permutation regression
            results = statistics_utils.permutation_regression_zscore(chan_df_band, 'tfr ~ 1 + zrpe + zpm', n_permutations=1000, plot_res=False)
            model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
            model_results['pm'].append(results.loc['zpm']['Z_Score'])

            memory_df = combined_df[(combined_df.condition == 'Day1') & (combined_df.participant == chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
            memory_df['trials'] = memory_df['trials_gamble']
            # code memory as 1,0
            memory_df['phit'] = np.where(memory_df.hits == True, 0.5, -0.5)
            chan_df = chan_df.merge(memory_df, on=['participant', 'trials'])
            chan_df_band = chan_df[chan_df.fband == band].dropna(subset=['tfr', 'phit'])

            results = statistics_utils.permutation_regression_zscore(chan_df_band, 'tfr ~ 1 + C(phit)*zrpe', n_permutations=1000, plot_res=False)
            model_results['phit'].append(results.loc['C(phit)[T.0.5]']['Z_Score'])
            model_results['phit:zrpe'].append(results.loc['C(phit)[T.0.5]:zrpe']['Z_Score'])
            
            # Fill out other data
            model_results['participant'].append(chan_df.participant.unique()[0])
            model_results['channel'].append(chan_df.channel.unique()[0])
            model_results['region'].append(chan_df.region.unique()[0])
            # model_results['hemi'].append(chan_df.hemi.str.lower().unique()[0])
            model_results['unique_elec_id'].append(chan_df.unique_elec_id.unique()[0])

            elec_counter += 1

    gamble_model_df = pd.DataFrame(model_results)
    gamble_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx', index=False)


In [ ]:
combined_df[(combined_df.condition == 'Day1') & (combined_df.participant == chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]

In [ ]:
gamble_model_df

In [ ]:
# sns.lmplot(data=chan_df_band, x='zrpe', y='tfr', hue='phit')

## if separating by blocks: 

In [ ]:
rois = ['OFC']
bands = ['hfa']
analysis_evs = 'feedback_start'

blocks = [[0, 0.5], [0.5, 1], [1, 1.5]]

for band in bands: 
    
    for block in blocks:
        model_results = {'participant':[], 'channel':[], 'region':[], 'unique_elec_id':[], 'rpe':[], 'pm':[], 'phit':[], 'phit:zrpe':[]} 
        chan_dfs = [] 
        elec_counter = 0 
    
        for subj_id in subj_ids:

            print(f'Loading subj {subj_id}')
            day = 'Day1'
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

            # Pre-filter for ROIs and to omit lesion electrodes
            valid_elec_df = elec_df[elec_df.salman_region.isin(rois) & ~elec_df.label.str.contains('lesion')]

            for chan in valid_elec_df.label.unique():

                chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time{block[0]}-{block[1]}.csv')
                chan_df = chan_df[['region', 'channel', 'fband', 'trial', 'tfr']]
                chan_df.rename(columns={'trial':'trials'}, inplace=True) 
                chan_df['participant'] = subj_id
                chan_df['unique_elec_id'] = elec_counter
                
                chan_df = chan_df[['participant', 'region', 'channel', 'unique_elec_id', 'trials', 'fband', 'tfr']]

                chan_df = chan_df.merge(learn_df, on=['participant', 'trials'])

                zrpe_mean = np.nanmean(chan_df.rpe)
                zrpe_std = np.nanstd(chan_df.rpe)
                chan_df['zrpe'] = (chan_df.rpe - zrpe_mean) / (2 * zrpe_std)
                chan_df['urpe'] = np.abs(chan_df.rpe)

                zpm_mean = np.nanmean(chan_df.DPRIME)
                zpm_std = np.nanstd(chan_df.DPRIME)
                chan_df['zpm'] = (chan_df.DPRIME - zpm_mean) / (2 * zpm_std)

                chan_df_band = chan_df[chan_df.fband == band]
                chan_df_band = chan_df_band.dropna(subset=['tfr', 'zrpe', 'zpm'])

                # Run permutation regression
                results = statistics_utils.permutation_regression_zscore(chan_df_band, 'tfr ~ 1 + zrpe + zpm', n_permutations=250, plot_res=False)
                model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
                model_results['pm'].append(results.loc['zpm']['Z_Score'])

                memory_df = combined_df[(combined_df.condition == 'Day1') & (combined_df.participant == chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
                memory_df['trials'] = memory_df['trials_gamble']
                # code memory as 1,0
                memory_df['phit'] = np.where(memory_df.hits == True, 0.5, -0.5)
                chan_df = chan_df.merge(memory_df, on=['participant', 'trials'])
                chan_df_band = chan_df[chan_df.fband == band].dropna(subset=['tfr', 'phit'])

                results = statistics_utils.permutation_regression_zscore(chan_df_band, 'tfr ~ 1 + phit*zrpe', n_permutations=250, plot_res=False)
                model_results['phit'].append(results.loc['phit']['Z_Score'])
                model_results['phit:zrpe'].append(results.loc['phit:zrpe']['Z_Score'])

                # Fill out other data
                model_results['participant'].append(chan_df.participant.unique()[0])
                model_results['channel'].append(chan_df.channel.unique()[0])
                model_results['region'].append(chan_df.region.unique()[0])
                model_results['unique_elec_id'].append(chan_df.unique_elec_id.unique()[0])

                elec_counter += 1

        gamble_model_df = pd.DataFrame(model_results)
        gamble_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx_{block[0]}-{block[1]}', index=False)


In [ ]:
gamble_model_dfs = []
counter = 0
for block in blocks:
    counter += 1
    gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx_{block[0]}-{block[1]}')
    gamble_model_df['block'] = counter
    gamble_model_dfs.append(gamble_model_df) 


In [ ]:
test_df = pd.concat(gamble_model_dfs)

In [ ]:
gamble_model_df[gamble_model_df.region=='OFC']

In [ ]:
sns.regplot(x=test_df.loc[test_df.block==2, 'phit:zrpe'],
            y=test_df.loc[test_df.block==3, 'phit:zrpe'])

In [ ]:
pearsonr(test_df.loc[test_df.block==1, 'phit:zrpe'], 
         test_df.loc[test_df.block==2, 'phit:zrpe'])

In [ ]:
sns.regplot(x=test_df.loc[test_df.block==3, 'phit:zrpe'],
            y=gamble_model_df[gamble_model_df.region=='OFC']['phit:zrpe'])

There is not as much overlap between time-periods as you might think, and very little between the component time blocks and the overall 1.5 seconds

In [ ]:
test_df.loc[test_df.block==2, 'phit:zrpe']

In [ ]:
test_df.loc[test_df.block==1, 'phit:zrpe']

In [ ]:
# for block in blocks:
#     gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx_{block[0]}-{block[1]}')
#     fig, ax = plt.subplots(1, 1, figsize=(3, 5), dpi=300)
#     sns.despine()
#     # with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_phit_intx_eleclevel.pdf') as pdf:
#     sns.barplot(data=gamble_model_df, x='region', y='rpe',errorbar='se', 
#             order = ['dmPFC', 'dlPFC', 'HPC', 'OFC', 'AMY'], alpha = 0.5, color='k')
#     sns.stripplot(data=gamble_model_df[np.abs(gamble_model_df['rpe'])<=1.96], x='region', y='rpe', 
#         order = ['dmPFC', 'dlPFC', 'HPC','OFC', 'AMY'], alpha=0.1, color='k')
#     sns.stripplot(data=gamble_model_df[np.abs(gamble_model_df['rpe'])>1.96], x='region', y='rpe', 
#         order = ['dmPFC', 'dlPFC', 'HPC', 'OFC', 'AMY'], alpha=0.5, color='red')

#     plt.hlines([0], -1, 4, 'k')
#     ax.set(ylabel='RPE sensitivity', xlim=[-1,4], ylim=[-3.5, 3.5])
    

In [ ]:
# for block in blocks:
#     gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_rpe_pm_phit_intx_eleclevel_df_time{block[0]}-{block[1]}')
#     fig, ax = plt.subplots(1, 1, figsize=(3, 5), dpi=300)
#     sns.despine()
#     # with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_phit_intx_eleclevel.pdf') as pdf:
#     sns.barplot(data=gamble_model_df, x='region', y='pm',errorbar='se', 
#             order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha = 0.5, color='k')
#     sns.stripplot(data=gamble_model_df, x='region', y='pm', 
#         order = ['dmPFC', 'HPC','OFC', 'AMY'], alpha=0.1, color='k')
#     sns.stripplot(data=gamble_model_df[np.abs(gamble_model_df['pm'])>1.96], x='region', y='pm', 
#         order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.5, color='red')

#     plt.hlines([0], -1, 4, 'k')
#     ax.set(ylabel='PM sensitivity', xlim=[-1,4], ylim=[-3.5, 3.5])
    

In [ ]:
for block in blocks:
    gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx_{block[0]}-{block[1]}')
    fig, ax = plt.subplots(1, 1, figsize=(3, 5), dpi=300)
    sns.despine()
    # with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_phit_intx_eleclevel.pdf') as pdf:
    sns.barplot(data=gamble_model_df, x='region', y='phit:zrpe',errorbar='se', 
            order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=gamble_model_df, x='region', y='phit:zrpe', 
        order = ['dmPFC', 'HPC','OFC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=gamble_model_df[np.abs(gamble_model_df['phit:zrpe'])>1.96], x='region', y='phit:zrpe', 
        order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.5, color='red')

    plt.hlines([0], -1, 4, 'k')
    ax.set(ylabel='RPE:PHIT sensitivity', xlim=[-1,4], ylim=[-3.5, 3.5])
    

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(3, 5), dpi=300)
# sns.despine()
# # with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_phit_intx_eleclevel.pdf') as pdf:
# sns.barplot(data=gamble_model_df, x='region', y='phit:rpe',errorbar='se', 
#         order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha = 0.5, color='k')
# sns.stripplot(data=gamble_model_df, x='region', y='phit:rpe', 
#     order = ['dmPFC', 'HPC','OFC', 'AMY'], alpha=0.1, color='k')
# sns.stripplot(data=gamble_model_df[np.abs(gamble_model_df['phit:rpe'])>1.96], x='region', y='phit:rpe', 
#     order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.5, color='red')

# plt.hlines([0], -1, 4, 'k')
# ax.set(ylabel='RPE:PHIT sensitivity', xlim=[-1,4], ylim=[-3.5, 3.5])
#     # pdf.savefig(fig)

In [ ]:
# # Are the region-grouped beta values different from 0?
# p_values = [] 
# for region in ['dmPFC', 'HPC', 'OFC', 'AMY']: 
#     p_values.append(ttest_1samp(gamble_model_df[gamble_model_df.region==region]['phit:rpe'].values, popmean=0)[1])
    

# [x*4 for x in p_values]

9/23/24: Convert this to bambi, plot posteriors, and save out for supplement

## load the data and make some plots

In [ ]:
band = 'high_theta'
analysis_evs = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')




What regions encodes positive RPE?

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_eleclevel.pdf') as pdf:
    sns.barplot(data=gamble_model_df, x='region', y='rpe',errorbar='se', 
            order = ['dmPFC',  'dlPFC', 'OFC',  'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=gamble_model_df[(np.abs(gamble_model_df.rpe)<=1.96)], x='region', y='rpe', 
        order = ['dmPFC', 'dlPFC', 'OFC','HPC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=gamble_model_df[(np.abs(gamble_model_df.rpe)>1.96)], x='region', y='rpe', 
        order = ['dmPFC',  'dlPFC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='red')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='RPE sensitivity', xlim=[-1, 5], ylim=[-3.75, 3.75])
    pdf.savefig(fig)

Is this significant? (mixed-effects model)

In [ ]:
region = 'OFC'
predictor = 'rpe'

if predictor == 'phit:zrpe':
    # rename this for our model
    gamble_model_df.rename(columns={'phit:zrpe':'phit_rpe_intx'}, inplace=True)
    predictor = 'phit_rpe_intx'

model = bmb.Model(f'{predictor} ~ 1 + (1|participant)', 
                  gamble_model_df[gamble_model_df.region==region], 
                  family='gaussian', categorical=['participant'])

# Fit the model
results = model.fit(draws=1000, chains=4)

df = az.summary(results)

# if results.pvalues['Intercept']<=0.025:
if np.sign(df.loc['Intercept']['hdi_3%']) == np.sign(df.loc['Intercept']['hdi_97%']):
    
    # print(results.summary())
    with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_{region}_{predictor}_M_on_betas.pdf') as pdf:

        # Create a scatter plot of individual data points
        fig, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
        sns.stripplot(x="participant", y=f'{predictor}', data=gamble_model_df[gamble_model_df.region==region], 
                      jitter=True, alpha=0.6, color="gray", s=6, ax=ax)

        # plot the mean line
        subject_means = gamble_model_df[gamble_model_df.region==region].groupby("participant")[f'{predictor}'].mean()

        sns.boxplot(showmeans=True,
                    meanline=True,
                    meanprops={'color': 'k', 'ls': '-', 'lw': 2},
                    medianprops={'visible': False},
                    whiskerprops={'visible': False},
                    zorder=10,
                    x=subject_means.index,
                    y=subject_means.values,
                    showfliers=False,
                    showbox=False,
                    showcaps=False,
                    ax=ax)


        # # Overlay the subject means
        overall_mean = df.loc['Intercept']['mean']
        # Plot the overall mean from the mixed-effects model
        plt.axhline(overall_mean, color="blue", linestyle="--", label=f"Overall Mean: {overall_mean:.2f}")
        plt.axhline(df.loc['Intercept']['hdi_3%'], color='red', linestyle='--')
        plt.axhline(df.loc['Intercept']['hdi_97%'], color='red', linestyle='--')

        plt.axhline(0, color='black')
        sns.despine()
        
        pdf.savefig()

In [ ]:
df

What regions encoding the interaction of positive RPE with memory?

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_phit:rpe_eleclevel.pdf') as pdf:
    sns.barplot(data=gamble_model_df, x='region', y='phit:zrpe',errorbar='se', 
            order = ['dmPFC',  'dlPFC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=gamble_model_df[(np.abs(gamble_model_df['phit:zrpe'])<=1.96)], x='region', y='phit:zrpe', 
        order = ['dmPFC', 'dlPFC', 'OFC','HPC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=gamble_model_df[(np.abs(gamble_model_df['phit:zrpe'])>1.96)], x='region', y='phit:zrpe', 
        order = ['dmPFC',  'dlPFC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='red')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='RPE:memory interaction', xlim=[-1,5], ylim=[-3.75, 3.75])
    pdf.savefig(fig)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_pm_eleclevel.pdf') as pdf:
    sns.barplot(data=gamble_model_df, x='region', y='pm',errorbar='se', 
            order = ['dmPFC',  'dlPFC', 'OFC',  'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=gamble_model_df[(np.abs(gamble_model_df.pm)<=1.96)], x='region', y='pm', 
        order = ['dmPFC', 'dlPFC', 'OFC','HPC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=gamble_model_df[(np.abs(gamble_model_df.pm)>1.96)], x='region', y='pm', 
        order = ['dmPFC',  'dlPFC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='red')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='PM sensitivity', xlim=[-1, 5], ylim=[-3.75, 3.75])
    pdf.savefig(fig)

In [ ]:
region = 'dlPFC'
predictor = 'pm'

if predictor == 'phit:zrpe':
    # rename this for our model
    gamble_model_df.rename(columns={'phit:zrpe':'phit_rpe_intx'}, inplace=True)
    predictor = 'phit_rpe_intx'

model = bmb.Model(f'{predictor} ~ 1 + (1|participant)', 
                  gamble_model_df[gamble_model_df.region==region], 
                  family='gaussian', categorical=['participant'])

# Fit the model
results = model.fit(draws=1000, chains=4)

df = az.summary(results)

# if results.pvalues['Intercept']<=0.025:
if np.sign(df.loc['Intercept']['hdi_3%']) == np.sign(df.loc['Intercept']['hdi_97%']):
    
    # print(results.summary())
    with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_{region}_{predictor}_M_on_betas.pdf') as pdf:

        # Create a scatter plot of individual data points
        fig, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
        sns.stripplot(x="participant", y=f'{predictor}', data=gamble_model_df[gamble_model_df.region==region], 
                      jitter=True, alpha=0.6, color="gray", s=6, ax=ax)

        # plot the mean line
        subject_means = gamble_model_df[gamble_model_df.region==region].groupby("participant")[f'{predictor}'].mean()

        sns.boxplot(showmeans=True,
                    meanline=True,
                    meanprops={'color': 'k', 'ls': '-', 'lw': 2},
                    medianprops={'visible': False},
                    whiskerprops={'visible': False},
                    zorder=10,
                    x=subject_means.index,
                    y=subject_means.values,
                    showfliers=False,
                    showbox=False,
                    showcaps=False,
                    ax=ax)


        # # Overlay the subject means
        overall_mean = df.loc['Intercept']['mean']
        # Plot the overall mean from the mixed-effects model
        plt.axhline(overall_mean, color="blue", linestyle="--", label=f"Overall Mean: {overall_mean:.2f}")
        plt.axhline(df.loc['Intercept']['hdi_3%'], color='red', linestyle='--')
        plt.axhline(df.loc['Intercept']['hdi_97%'], color='red', linestyle='--')

        plt.axhline(0, color='black')
        sns.despine()
        
        pdf.savefig()

In [ ]:
    sns.barplot(data=gamble_model_df, x='region', y='phit:zrpe',errorbar='se', 
            order = ['dmPFC',  'dlPFC', 'OFC', 'ACC', 'HPC', 'AMY'], alpha = 0.5, color='k')

Is this significant? (mixed-effects model)

In [ ]:
region = 'OFC'
predictor = 'phit:zrpe'

if predictor == 'phit:zrpe':
    # rename this for our model
    gamble_model_df.rename(columns={'phit:zrpe':'phit_rpe_intx'}, inplace=True)
    predictor = 'phit_rpe_intx'

model = bmb.Model(f'{predictor} ~ 1 + (1|participant)', 
                  gamble_model_df[gamble_model_df.region==region], 
                  family='gaussian', categorical=['participant'])

# Fit the model
results = model.fit(draws=1000, chains=4)

df = az.summary(results)

# if results.pvalues['Intercept']<=0.025:
if np.sign(df.loc['Intercept']['hdi_3%']) == np.sign(df.loc['Intercept']['hdi_97%']):
    
    # print(results.summary())
    with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_{region}_{predictor}_M_on_betas.pdf') as pdf:

        # Create a scatter plot of individual data points
        fig, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
        sns.stripplot(x="participant", y=f'{predictor}', data=gamble_model_df[gamble_model_df.region==region], 
                      jitter=True, alpha=0.6, color="gray", s=6, ax=ax)

        # plot the mean line
        subject_means = gamble_model_df[gamble_model_df.region==region].groupby("participant")[f'{predictor}'].mean()

        sns.boxplot(showmeans=True,
                    meanline=True,
                    meanprops={'color': 'k', 'ls': '-', 'lw': 2},
                    medianprops={'visible': False},
                    whiskerprops={'visible': False},
                    zorder=10,
                    x=subject_means.index,
                    y=subject_means.values,
                    showfliers=False,
                    showbox=False,
                    showcaps=False,
                    ax=ax)


        # # Overlay the subject means
        overall_mean = df.loc['Intercept']['mean']
        # Plot the overall mean from the mixed-effects model
        plt.axhline(overall_mean, color="blue", linestyle="--", label=f"Overall Mean: {overall_mean:.2f}")
        plt.axhline(df.loc['Intercept']['hdi_3%'], color='red', linestyle='--')
        plt.axhline(df.loc['Intercept']['hdi_97%'], color='red', linestyle='--')

        plt.axhline(0, color='black')
        sns.despine()
        
        pdf.savefig()

In [ ]:
df

Plot the relationship between encoding activation for RPE:phit and retrieval activation for hit vs. miss: 

In [ ]:
gamble_model_df.rename(columns={'channel':'label'}, inplace=True)

In [ ]:
full_data_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_multinomial_df') 
comp_df = full_data_df[(full_data_df.category=='miss') & (full_data_df.region=='OFC')].merge(gamble_model_df[gamble_model_df.region=='OFC'], on=['participant', 'label'])



In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 3), dpi=300)
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/compare_encoding_cue_electrodes.pdf') as pdf:
    sns.regplot(data=comp_df, x='rpe', y='hit_compared_to_coefficient', color='k')
    sns.despine()
    pdf.savefig()

In [ ]:
results = smf.mixedlm('hit_compared_to_coefficient ~ 1 + rpe', data=comp_df, 
                      re_formula = '1 + rpe', 
                      groups='participant').fit()

In [ ]:
results.summary()

Plot the relationship between RPE, TFR, and memory for OFC HFA

In [ ]:
rois = ['OFC']
band = 'hfa'

elec_counter = 0 
chan_dfs = []
for subj_id in subj_ids:
    print(f'Loading subj {subj_id}')
    day = 'Day1'
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

    for chan in elec_df.label.unique():
        if 'lesion' in chan:
            # skip the lesion channels (dmPFC)
            print('skipping lesion channel')
            continue
        if elec_df[elec_df.label==chan].salman_region.values[0] in rois:
            chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time0-1.5.csv') # _time{block[0]}-{block[1]}
            chan_df['unique_elec_id'] = elec_counter
            chan_df = chan_df[['participant', 'region', 'channel', 'unique_elec_id', 'trials', 'fband', 'tfr']]

            chan_df = chan_df.merge(learn_df, on=['participant', 'trials'])

            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
            chan_df['binarized_rpe'] = chan_df.rpe.apply(lambda x: 1 if x>0 else -1)
            chan_df['urpe'] = np.abs(chan_df.rpe)
            chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
            chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
            
            memory_df = combined_df[(combined_df.condition == 'Day1') & (combined_df.participant == chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
            memory_df['trials'] = memory_df['trials_gamble']
            # code memory as 1,0
            memory_df['phit'] = np.where(memory_df.hits == True, 0.5, -0.5)
            chan_df = chan_df.merge(memory_df, on=['participant', 'trials'])
            chan_df_band = chan_df[chan_df.fband == band].dropna(subset=['tfr', 'phit'])
            
            elec_counter +=1
            chan_dfs.append(chan_df)
            
chan_dfs = pd.concat(chan_dfs)
chan_dfs = chan_dfs[chan_dfs.fband==band]

In [ ]:
# rename this for our model
gamble_model_df.rename(columns={'phit:zrpe':'phit_rpe_intx'}, inplace=True)
predictor = 'phit_rpe_intx'
sig_elecs = gamble_model_df[gamble_model_df.phit_rpe_intx<-2][['participant', 'channel']]


In [ ]:
gamble_model_df[gamble_model_df.phit_rpe_intx<-2]

In [ ]:
gamble_model_df[gamble_model_df.phit_rpe_intx>3]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 3), dpi=300)
sns.despine()
ex_subj = 'UI006'
ex_channel = 'lfpx33-lfpx34'
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/example_OFC_{ex_subj}_{ex_channel}_phit:rpe.pdf') as pdf:
    sns.regplot(data=chan_dfs[(chan_dfs.fband=='hfa') & (chan_dfs.participant==ex_subj) & (chan_dfs.channel==ex_channel) & (chan_dfs.hits==False)],
               x='zrpe', y='tfr',  ci=None, color='gray', ax=ax)
    
    sns.regplot(data=chan_dfs[(chan_dfs.fband=='hfa') & (chan_dfs.participant==ex_subj) & (chan_dfs.channel==ex_channel) & (chan_dfs.hits==True)],
               x='zrpe', y='tfr',  ci=None, color='black', ax=ax)
    
    pdf.savefig()
    
    

## What is the relationship between encoding-related HPC input and OFC rpe:phit coding

In [ ]:
def compute_auc_2d(array_2d, timepoints=None):
    # If no timepoints are provided, assume uniform timepoints
    if timepoints is None:
        timepoints = np.arange(array_2d.shape[1])
    
    positive_aucs = []
    negative_aucs = []
    
    for timeseries in array_2d:
        # Compute positive and negative AUC for the current timeseries
        positive_values = np.clip(timeseries, a_min=0, a_max=None)
        negative_values = np.clip(timeseries, a_min=None, a_max=0)
        
        # Compute AUC using the trapezoidal rule
        positive_auc = np.trapz(positive_values, timepoints)
        negative_auc = np.trapz(negative_values, timepoints)
        
        # Store results
        positive_aucs.append(positive_auc)
        negative_aucs.append(negative_auc)
    
    return np.array(positive_aucs), np.array(negative_aucs)

In [ ]:
band = 'hfa'
analysis_evs = 'feedback_start'

if analysis_evs == 'feedback_start':
    gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')
else:
    gamble_model_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_multinomial_df_wrpe') 
    
gamble_model_df.rename(columns={'channel':'label'}, inplace=True)


source_region = 'HPC'

band1= 'high_theta'
band2= 'high_theta'
event = 'feedback_start'
metric = 'psi'

all_meta_data = [] 
for target_region in ['OFC']:
    conn_group_data_pos = np.load(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/psi/{source_region}_{target_region}_psi_{band1}_{band2}_{event}_hit_Z.npy')
    meta_data = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/psi/{source_region}_{target_region}_psi_{band1}_{band2}_{event}_hit_Z_df')
    meta_data.hemi = meta_data.hemi.str.lower()
    if analysis_evs == 'feedback_start':
        times = np.linspace(0, 1500, conn_group_data_pos[:, :].shape[1])
    else:
        times = np.linspace(0, 500, conn_group_data_pos[:, :].shape[1])
    meta_data['psi'] = np.nanmean(conn_group_data_pos[:, :], axis=1) 
    meta_data['cond'] = 'hit'
    source_to_target_auc, target_to_source_auc = compute_auc_2d(conn_group_data_pos)
    meta_data[f'{source_region}_{target_region}_auc'] = source_to_target_auc
    meta_data[f'{target_region}_{source_region}_auc'] = target_to_source_auc
    meta_data['label'] = meta_data.pair_labels.apply(lambda x: x.split(':')[1])
    all_meta_data.append(meta_data) 
    
for target_region in ['OFC']:
    conn_group_data_neg = np.load(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/psi/{source_region}_{target_region}_psi_{band1}_{band2}_{event}_miss_Z.npy')
    meta_data = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/psi/{source_region}_{target_region}_psi_{band1}_{band2}_{event}_miss_Z_df')
    meta_data.hemi = meta_data.hemi.str.lower()
    if analysis_evs == 'feedback_start':
        times = np.linspace(0, 1500, conn_group_data_pos[:, :].shape[1])
    else:
        times = np.linspace(0, 500, conn_group_data_pos[:, :].shape[1])
    meta_data['psi'] = np.nanmean(conn_group_data_neg[:, :], axis=1) 
    meta_data['cond'] = 'miss'
    source_to_target_auc, target_to_source_auc = compute_auc_2d(conn_group_data_neg)
    meta_data[f'{source_region}_{target_region}_auc'] = source_to_target_auc
    meta_data[f'{target_region}_{source_region}_auc'] = target_to_source_auc
    meta_data['label'] = meta_data.pair_labels.apply(lambda x: x.split(':')[1])
    all_meta_data.append(meta_data) 
    
# for target_region in ['OFC']:
#     conn_group_data_pos = np.load(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/psi/{source_region}_{target_region}_psi_{band1}_{band2}_{event}_hit_Z.npy')
#     conn_group_data_neg = np.load(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/psi/{source_region}_{target_region}_psi_{band1}_{band2}_{event}_miss_Z.npy')

#     meta_data = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/psi/{source_region}_{target_region}_psi_{band1}_{band2}_{event}_hit_Z_df')
#     meta_data.hemi = meta_data.hemi.str.lower()
#     times = np.linspace(0, 1500, conn_group_data_pos[:, :].shape[1])
#     meta_data['psi'] = np.nanmean(conn_group_data_pos[:, (times>=750)], axis=1) 
#     meta_data['cond'] = 'late'
#     meta_data['label'] = meta_data.pair_labels.apply(lambda x: x.split(':')[1])
#     # all_meta_data.append(meta_data) 

    
all_meta_data = pd.concat(all_meta_data)
# mean the data 
mean_psi = meta_data.groupby(['participant', 'label']).mean(numeric_only=True).reset_index()
# max_psi = meta_data.loc[meta_data.groupby(['participant', 'label', 'roi1'])['psi'].idxmax()]

# merge_df = gamble_model_df[gamble_model_df.region=='OFC'].merge(mean_psi,
                                                    # on=['participant', 'label'])
merge_df = gamble_model_df[gamble_model_df.region=='OFC'].merge(all_meta_data,
                                                    on=['participant', 'label'])

merge_df['unique_elec_label'] =  merge_df['participant'] + '_' + merge_df['label']

# sns.regplot(data=merge_df, x='psi', y='phit:zrpe')

# predictor = 'hit_compared_to_coefficient'
predictor = 'phit:zrpe'

if predictor == 'phit:zrpe':
    # rename this for our model
    merge_df.rename(columns={'phit:zrpe':'phit_rpe_intx'}, inplace=True)
    predictor = 'phit_rpe_intx'

model = bmb.Model(f'{predictor} ~ 1 + psi*cond + (1|participant)', 
                  merge_df, 
                  family='gaussian', categorical=['cond', 'participant'])

# Fit the modelb

results = model.fit(draws=1000, chains=4, target_accept=0.9)

df = az.summary(results)

In [ ]:
merge_df[merge_df.cond=='hit'].sort_values(by=['phit_rpe_intx', 'psi'], ascending=False)

In [ ]:
df[df.index=='psi:cond[miss]']

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 3), dpi=300)


with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/full_hits_psi_rpe:phit_AMYOFC_high_theta.pdf') as pdf:


    sns.regplot(x='psi', y='phit_rpe_intx', data=merge_df[merge_df.cond=='hit'], scatter=False, ci=95, color ='black')
    # sns.regplot(x='psi', y='phit_rpe_intx', data=merge_df[merge_df.cond=='miss'], scatter=False, ci=95, color ='k')

    for label in merge_df['participant'].unique():
        # Get the data for this participant
        df_part = merge_df[(merge_df.cond=='hit') & (merge_df['participant'] == label)]

        # Fit linear model (intercept-only random effects like)
        X = sm.add_constant(df_part['psi'])  # Adding constant for intercept
        model = sm.OLS(df_part['phit_rpe_intx'], X).fit()

        # Get the predicted values
        df_part['fitted'] = model.predict(X)

        # Plot the fitted line for this subject
        # plt.scatter(df_part['psi'], df_part['phit_rpe_intx'], color = 'orange', alpha=0.3)
        sns.regplot(data=df_part, x='psi', y='fitted', line_kws={'alpha':0.2}, color = 'black',scatter=False, ci=None)

    sns.despine()

    ax.set(xlabel='theta PSI (z)', ylabel='rpe:encoding modulation (z)', ylim=[-1, 2.5])
    
    pdf.savefig()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 3), dpi=300)

with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/full_miss_psi_rpe:phit_AMYCOFC_high_theta.pdf') as pdf:


    sns.regplot(x='psi', y='phit_rpe_intx', data=merge_df[merge_df.cond=='miss'], scatter=False, ci=95, color ='gray')
    # sns.regplot(x='psi', y='phit_rpe_intx', data=merge_df[merge_df.cond=='miss'], scatter=False, ci=95, color ='k')

    for label in merge_df['participant'].unique():
        # Get the data for this participant
        df_part = merge_df[(merge_df.cond=='miss') & (merge_df['participant'] == label)]

        # Fit linear model (intercept-only random effects like)
        X = sm.add_constant(df_part['psi'])  # Adding constant for intercept
        model = sm.OLS(df_part['phit_rpe_intx'], X).fit()

        # Get the predicted values
        df_part['fitted'] = model.predict(X)

        # Plot the fitted line for this subject
        # plt.scatter(df_part['psi'], df_part['phit_rpe_intx'], color = 'orange', alpha=0.3)
        sns.regplot(data=df_part, x='psi', y='fitted', line_kws={'alpha':0.2}, color = 'gray',scatter=False, ci=None)

    sns.despine()

    ax.set(xlabel='theta PSI (z)', ylabel='rpe:encoding modulation (z)', ylim=[-1, 2.5])
    
    pdf.savefig()

In [ ]:
# Create a scatterplot and linear fit for overall data

plot_df = merge_df[merge_df.cond=='hit']

plt.figure(figsize=(4, 4))
sns.regplot(x='psi', y='phit_rpe_intx', data=plot_df, scatter=False, label='Pooled Fit', color ='k')

# Now plot individual fits per subject
labels = plot_df['participant'].unique()

for label in labels:
    # Get the data for this participant
    df_part = merge_df[merge_df['participant'] == label]
    
    # Fit linear model (intercept-only random effects like)
    X = sm.add_constant(df_part['psi'])  # Adding constant for intercept
    model = sm.OLS(df_part['phit_rpe_intx'], X).fit()
    
    # Get the predicted values
    df_part['fitted'] = model.predict(X)
    
    # Plot the fitted line for this subject
    # plt.scatter(df_part['psi'], df_part['phit_rpe_intx'], color = 'orange', alpha=0.3)
    plt.plot(df_part['psi'], df_part['fitted'], color = 'purple', alpha=0.3)

plt.xlabel('psi')
plt.ylabel('phit_rpe_intx')
plt.title('Pooled vs. Subject-Specific Linear Fits')
plt.legend(loc='best')
plt.show()

## Can I persuade that positive RPE hits have more HFA than positive RPE misses in OFC?

In [ ]:
# LOAD the data
analysis_evs = ['feedback_start']
chan_dfs = [] 
glms = []
elec_counter = 0 
rois = ['OFC']
for subj_id in subj_ids:
    print(f'Loading subj {subj_id}')
    day = 'Day1'
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

    # Pre-filter for ROIs and to omit lesion electrodes
    valid_elec_df = elec_df[elec_df.salman_region.isin(rois) & ~elec_df.label.str.contains('lesion')]
    
    for event in analysis_evs:
        for chan in valid_elec_df.label.unique():
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')


            chan_df['unique_elec_id'] = elec_counter
            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
            chan_df['urpe'] = np.abs(chan_df.rpe)
            chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
            chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]

            memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
            memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
            memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
            chan_df = chan_df.merge(memory_df, on=['participant', 'trials'])

            elec_counter +=1

            chan_dfs.append(chan_df)

chan_dfs = pd.concat(chan_dfs)
chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))

In [ ]:
sns.pointplot(data=chan_dfs[chan_dfs.fband=='hfa'], 
               x='hits', y='tfr', errorbar=None, hue='channel', legend=None, alpha=0.1)

## Can we demonstrate that OFC rpe:phit during encoding is stronger than during the cue period? 

In [ ]:
band = 'hfa'
analysis_evs = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')

cue_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_rpe_pm_phit_intx_eleclevel_df')

# cue_model_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_multinomial_df_wrpe') 
# cue_model_df = cue_model_df[cue_model_df.category=='miss']
# cue_model_df = cue_model_df[['participant', 'label', 'hit_compared_to_coefficient']]
# cue_model_df.rename(columns={'hit_compared_to_coefficient':'phit:zrpe'}, inplace=True)


In [ ]:
cue_model_df[cue_model_df.region=='OFC']

In [ ]:
from scipy.stats import wilcoxon
wilcoxon(gamble_model_df[gamble_model_df.region=='OFC']['phit:zrpe'],
          cue_model_df[cue_model_df.region=='OFC']['phit:rpe'])

In [ ]:
gamble_model_df

## Is there any relationship between OFC theta RPE coding and OFC HFA phit:rpe coding:

In [ ]:
band = 'theta'
analysis_evs = 'feedback_start'
gamble_model_df1 = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')

band = 'hfa'
analysis_evs = 'feedback_start'
gamble_model_df2 = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')



In [ ]:
merged_df = gamble_model_df1.merge(gamble_model_df2, on=['participant', 'channel', 'region'], suffixes=['_theta', '_hfa'])

In [ ]:
merged_df.rename(columns={'phit:zrpe_hfa': 'phit_zrpe_hfa', 
                          'phit:zrpe_theta': 'phit_zrpe_theta'}, inplace=True)

In [ ]:
# Create a scatterplot and linear fit for overall data
plt.figure(figsize=(9, 6))
sns.regplot(x='pm_hfa', y='phit_zrpe_hfa', data=merged_df, scatter=False, label='Pooled Fit', color ='k')

# Now plot individual fits per subject
participants = merged_df['participant'].unique()

for participant in participants:
    # Get the data for this participant
    df_part = merged_df[merged_df['participant'] == participant]
    
    # Fit linear model (intercept-only random effects like)
    X = sm.add_constant(df_part['pm_hfa'])  # Adding constant for intercept
    model = sm.OLS(df_part['phit_zrpe_hfa'], X).fit()
    
    # Get the predicted values
    df_part['fitted'] = model.predict(X)
    
    # Plot the fitted line for this subject
    plt.scatter(df_part['pm_hfa'], df_part['phit_zrpe_hfa'], color = 'orange', alpha=0.3)
    plt.plot(df_part['pm_hfa'], df_part['fitted'], color = 'orange', alpha=0.3)

plt.xlabel('pm_hfa')
plt.ylabel('phit_zrpe_hfa')
plt.title('Pooled vs. Subject-Specific Linear Fits')
plt.legend(loc='best')
plt.show()

In [ ]:
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/OFC_pm_phit:rpe_hfa.pdf') as pdf:

    sns.lmplot(data=merged_df[merged_df.region=='OFC'], x='pm_hfa', y='phit_zrpe_hfa', hue='participant', ci=None, 
               palette=sns.color_palette(['orange'], len(merged_df['participant'].unique())), 
               scatter_kws={'alpha':0.2}, line_kws={'alpha':0.4}, height=5, legend=None)
    sns.regplot(data=merged_df[merged_df.region=='OFC'], x='pm_hfa', y='phit_zrpe_hfa', color='k', ci=95, scatter=None)

    plt.ylim([-4, 4])
    
    pdf.savefig()

In [ ]:
sns.regplot(data=merged_df[merged_df.region=='OFC'], x='pm_hfa', y='phit_zrpe_hfa', color='k', ci=95, scatter=None)
sns.regplot(data=merged_df[merged_df.region=='OFC'], x='pm_hfa', y='phit_zrpe_hfa', color='k', ci=95, scatter=None)


In [ ]:
sns.lmplot(data=merged_df[merged_df.region=='OFC'], x='rpe_theta', y='phit_zrpe_hfa', hue='participant', ci=None, 
           palette=sns.color_palette(['purple'], len(merged_df['participant'].unique())), 
           scatter_kws={'alpha':0.2}, line_kws={'alpha':0.2}, height=5, legend=None)
sns.regplot(data=merged_df[merged_df.region=='OFC'], x='rpe_theta', y='phit_zrpe_hfa', color='k', ci=95, scatter=None)

plt.ylim([-4, 4])

In [ ]:
results = smf.mixedlm(data=merged_df[merged_df.region=='OFC'], 
            formula = 'phit_zrpe_hfa ~ 1 + rpe_theta', 
            re_formula = '1',
            groups='participant').fit(reml=False)

In [ ]:
results.summary()

In [ ]:
# Extract coefficients and variance-covariance matrix
coefs = results.fe_params
vcov_matrix = results.cov_params()

# Calculate the difference between coefficients
diff = coefs["pm_hfa"] - coefs["rpe_theta"]

# Calculate the standard error of the difference
se_diff = np.sqrt(vcov_matrix.loc["pm_hfa", "rpe_theta"] + 
                  vcov_matrix.loc["rpe_theta", "rpe_theta"] - 
                  2 * vcov_matrix.loc["pm_hfa", "rpe_theta"])

# Calculate t-statistic
t_stat = diff / se_diff

# Calculate degrees of freedom (this is an approximation for mixed models)
df = results.df_resid

# Calculate p-value (two-tailed test)
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df))

In [ ]:
effect

In [ ]:
results.fittedvalues

In [ ]:
effects

In [ ]:
results.fittedvalues

No relationship between the amount an OFC electrode desynchronizes with RPE (via theta) and the amount it increases activity for RPE as a function of memory (via HFA)

## Is there any spatial relationship between OFC position and encoding of phit:rpe

In [ ]:
band = 'hfa'
region = 'OFC'
analysis_evs = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')
gamble_model_df = gamble_model_df[gamble_model_df.region==region]
gamble_model_df.rename(columns={'channel':'label'}, inplace=True) 



In [ ]:
all_elec_dfs = []
# subj_ids = ['UI001', 'UI002', 'UI003', 'UI004']
for subj_id in subj_ids: 
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
    elec_df['participant'] = subj_id
    all_elec_dfs.append(elec_df)
all_elec_dfs = pd.concat(all_elec_dfs).reset_index()

merged_elec_df = all_elec_dfs.merge(gamble_model_df[gamble_model_df.region==region], on=['participant', 'label'])

merged_elec_df.hemisphere = merged_elec_df.hemisphere.str.lower()

In [ ]:
merged_elec_df[['mni_x', 'mni_y', 'mni_z', 'phit:zrpe']].to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/encoding_hfa_OFC_elecs', index=False)



In [ ]:
half_q_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/half_q_df')

In [ ]:
merged_elec_df = all_elec_dfs.merge(half_q_df, on=['participant', 'label'])



In [ ]:
sns.regplot(data=merged_elec_df, x='mni_z', y='half_q')

In [ ]:
predictor = 'pm'

if predictor == 'phit:zrpe':
    # rename this for our model
    merged_elec_df.rename(columns={'phit:zrpe':'phit_rpe_intx'}, inplace=True)
    predictor = 'phit_rpe_intx'

model = bmb.Model(f'{predictor} ~ 1 + (1|participant/hemisphere)', 
                  merged_elec_df, 
                  family='gaussian', categorical=['hemisphere', 'participant'])

# Fit the model
results = model.fit(draws=1000, chains=4, target_accept=0.9)

df = az.summary(results)

# if results.pvalues['Intercept']<=0.025:
if np.sign(df.loc['Intercept']['hdi_3%']) == np.sign(df.loc['Intercept']['hdi_97%']):
    
    # print(results.summary())
    # with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_{region}_{predictor}_M_on_betas.pdf') as pdf:

    # Create a scatter plot of individual data points
    fig, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
    sns.stripplot(x="participant", y=f'{predictor}', data=gamble_model_df[gamble_model_df.region==region], 
                  jitter=True, alpha=0.6, color="gray", s=6, ax=ax)

    # plot the mean line
    subject_means = gamble_model_df[gamble_model_df.region==region].groupby("participant")[f'{predictor}'].mean()

    sns.boxplot(showmeans=True,
                meanline=True,
                meanprops={'color': 'k', 'ls': '-', 'lw': 2},
                medianprops={'visible': False},
                whiskerprops={'visible': False},
                zorder=10,
                x=subject_means.index,
                y=subject_means.values,
                showfliers=False,
                showbox=False,
                showcaps=False,
                ax=ax)


    # # Overlay the subject means
    overall_mean = df.loc['Intercept']['mean']
    # Plot the overall mean from the mixed-effects model
    plt.axhline(overall_mean, color="blue", linestyle="--", label=f"Overall Mean: {overall_mean:.2f}")
    plt.axhline(df.loc['Intercept']['hdi_3%'], color='red', linestyle='--')
    plt.axhline(df.loc['Intercept']['hdi_97%'], color='red', linestyle='--')

    plt.axhline(0, color='black')
    sns.despine()
        
    # pdf.savefig()

In [ ]:
# Create the figure and axis
fig, ax = plt.subplots(figsize=(9, 5))
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encogin_spatial_location_rpe_phit.pdf') as pdf:

    merged_elec_df['mni_z_bin'] = pd.cut(merged_elec_df['mni_z'], bins=5)
    merged_elec_df['mni_y_bin'] = pd.cut(merged_elec_df['mni_y'], bins=5)

    # Group by the binned coordinates and calculate the mean of 'phit' for each bin
    df_binned = merged_elec_df.groupby(['mni_z_bin', 'mni_y_bin'])['phit:zrpe'].mean().reset_index()

    # Pivot the dataframe for the heatmap
    heatmap_data = df_binned.pivot(columns="mni_y_bin", index="mni_z_bin", values="phit:zrpe")

    # Plot the heatmap on the provided axis
    sns.heatmap(heatmap_data, cmap="viridis", annot=False, ax=ax)

    # Customize the plot
    ax.set_title("Heatmap of Phit by Binned MNI coordinates")
    plt.xticks(rotation=45)

    ax.invert_yaxis()
    ax.invert_xaxis()
    # Show the plot
    plt.show()
    pdf.savefig(fig)


In [ ]:
merged_elec_df

In [ ]:
df_binned

In [ ]:
merged_elec_df[["mni_y", "mni_x", "phit:zrpe"]].reset_index()

## Identify the electrodes that most increase their connectivity with OFC during hits: 

The logic here is that maybe what ever boosts connectivity during hits is also what corresponds to increase the rpe:hfa slope during hits in OFC. 

In [ ]:
# First, let's get all the OFC electrodes and their phit:rpe slope: 

band = 'hfa'
analysis_evs = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')



## Main trialwise undirected connectivity analysis of theta connectivity (coh/wPLI): 

In [ ]:
# pairs = [['AMY', 'OFC'],['HPC', 'OFC'], ['dmPFC', 'OFC'], ['dlPFC', 'OFC'], ['ACC', 'OFC'],
#         ]

pairs = [['OFC', 'OFC'],]

metric = 'amp'
band1 = 'hfa'
band2 = 'hfa'
event = 'feedback_start' 
savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

all_dfs = [] 

for pair in pairs:
    
    all_pairs_df = pd.read_csv(f'{savepath}/{pair[0]}_{pair[1]}_{metric}_{band1}_{band2}_{event}_df')
    all_dfs.append(all_pairs_df) 
    
all_dfs = pd.concat(all_dfs) 

In [ ]:
all_dfs['label'] = all_dfs.pair_label.apply(lambda x: x.split(':')[1])

# Get the memory information: 
memory_df = combined_df[(combined_df.condition=='Day1')].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
memory_df['phit'] = np.where(memory_df.hits == True, 0.5, -0.5)
new_df = all_dfs.merge(memory_df, on=['participant', 'trials'])


In [ ]:
sns.regplot(data=new_df, x='rpe', y='conn', scatter=False)

In [ ]:
from scipy.stats import ttest_ind

In [ ]:
# Example: Assume `new_df` contains the data
results = []

# Group by participant and pair_label
grouped = new_df.groupby(['participant', 'pair_label'])

# Iterate through each group
for (participant, pair_label), group in grouped:
    # Subset the data into hits == True and hits == False
    hits_true = group[group.hits == True].conn
    hits_false = group[group.hits == False].conn
    
    # Perform t-test if both groups have data
    if len(hits_true) > 0 and len(hits_false) > 0:
        t_stat, p_value = ttest_ind(hits_true, hits_false, equal_var=False)  # Welch's t-test
        results.append({
            'participant': participant,
            'pair_label': pair_label,
            'roi1': group.roi1.unique()[0], 
            't_stat': t_stat,
            'p_value': p_value
        })

# Convert results into a DataFrame for further analysis or export
results_df = pd.DataFrame(results)
results_df['label'] = results_df.pair_label.apply(lambda x: x.split(':')[1])


In [ ]:
gamble_model_df.rename(columns={'channel': 'label'}, inplace=True)
gamble_model_df = gamble_model_df.merge(results_df, on=['participant', 'label'])

In [ ]:
gamble_model_df[gamble_model_df.p_value<=0.05].roi1.value_counts() / all_dfs.groupby(['participant','pair_label','roi1']).mean(numeric_only=True).reset_index().roi1.value_counts()



In [ ]:
gamble_model_df

In [ ]:
sns.regplot(data=gamble_model_df.groupby(['participant', 'label']).mean(numeric_only=True),
            x='t_stat', y='phit:zrpe')

In [ ]:
plot_df = gamble_model_df.groupby(['participant', 'label']).mean(numeric_only=True).reset_index()

In [ ]:


predictor = 'phit:zrpe'

if predictor == 'phit:zrpe':
    # rename this for our model
    plot_df.rename(columns={'phit:zrpe':'phit_rpe_intx'}, inplace=True)
    predictor = 'phit_rpe_intx'

model = bmb.Model(f'{predictor} ~ 1 + t_stat + (1|participant)', 
                  plot_df, 
                  family='gaussian', categorical=['participant'])

# Fit the model
results = model.fit(draws=1000, chains=4)

df = az.summary(results)


In [ ]:
df

In [ ]:
pairs = [['AMY', 'OFC'],['HPC', 'OFC'], ['dmPFC', 'OFC'], ['dlPFC', 'OFC'], ['OFC', 'OFC'],
        ]

# pairs = [['AMY', 'OFC'], ]
metric = 'coh'
band1 = 'theta'
band2 = 'theta'
event = 'feedback_start' 
savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

for pair in pairs:
    
    all_pairs_df = pd.read_csv(f'{savepath}/{pair[0]}_{pair[1]}_{metric}_{band1}_{band2}_{event}_df')

    # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 
    
    subj_list = all_pairs_df[['participant', 'pair_label']].drop_duplicates()['participant'].values
    pair_list = all_pairs_df[['participant', 'pair_label']].drop_duplicates()['pair_label'].values

    model_results = {'participant':[], 'pair_label':[], 'roi1':[], 'roi2':[], 'unique_pair_id':[], 
                     'rpe':[], 'pm':[], 'phit':[]} 
    pair_counter = 0
    
    for ix, _ in enumerate(pair_list):
        conn_df = all_pairs_df[(all_pairs_df.participant==subj_list[ix]) & (all_pairs_df.pair_label==pair_list[ix])]

        # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 

        if len(conn_df)>80:
            conn_df = conn_df.iloc[0:80]

        conn_df['unique_pair_id'] = pair_counter
        conn_df['zrpe'] = (conn_df.rpe - np.nanmean(conn_df.rpe)) / (2*np.nanstd(conn_df.rpe))
        conn_df['urpe'] = np.abs(conn_df.rpe)
        conn_df['zrpe_unsigned'] = (conn_df.urpe - np.nanmean(conn_df.urpe)) / (2*np.nanstd(conn_df.urpe))
        conn_df['zpm'] = (conn_df.DPRIME - np.nanmean(conn_df.DPRIME)) / (2*np.nanstd(conn_df.DPRIME))

        model_results['participant'].append(conn_df.participant.unique()[0])
        model_results['pair_label'].append(conn_df.pair_label.unique()[0])
        model_results['roi1'].append(conn_df.roi1.unique()[0])
        model_results['roi2'].append(conn_df.roi2.unique()[0])
        model_results['unique_pair_id'].append(conn_df.unique_pair_id.unique()[0])

        formula = 'conn ~ 1 + zrpe + zpm'
        results = statistics_utils.permutation_regression_zscore(conn_df, 
                                                formula, 
                                                n_permutations=250, 
                                                plot_res=False)

        model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
        model_results['pm'].append(results.loc['zpm']['Z_Score'])

        memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_list[ix])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
        memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
        memory_df['phit'] = np.where(memory_df.hits == True, 0.5, -0.5)
        new_df = conn_df.merge(memory_df, on=['participant', 'trials'])

        formula = 'conn ~ 1 + phit'

        results = statistics_utils.permutation_regression_zscore(new_df, 
                                                                 'conn ~ 1 + phit', n_permutations=250, plot_res=False)

        model_results['phit'].append(results.loc['phit']['Z_Score'])
        # model_results['phit:zrpe'].append(results.loc['phit']['Z_Score'])

        pair_counter += 1

    gamble_model_df = pd.DataFrame(model_results)
    gamble_model_df['paired_regions'] = f'{pair[0]}_{pair[1]}'
    gamble_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{pair[0]}_{pair[1]}_{band1}_{metric}_{event}_pair_level_df', index=False)


In [ ]:
gamble_model_df['label'] = gamble_model_df.pair_label.apply(lambda x: x.split(':')[1])


In [ ]:
df_filtered = gamble_model_df.loc[gamble_model_df.groupby(['participant', 'label'])['phit'].idxmax()]

In [ ]:
sns.barplot(data=df_filtered, x='roi1', y='phit')

In [ ]:
pairs = [['AMY', 'OFC'],['HPC', 'OFC'], ['dmPFC', 'OFC'], ['dlPFC', 'OFC'], ['OFC', 'OFC']
        ]

metric = 'coh'
band1 = 'theta'
event = 'feedback_start'

full_data_df = []
for pair in pairs:
    test_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{pair[0]}_{pair[1]}_{band1}_{metric}_{event}_pair_level_df')

    # test_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{pair[0]}_{pair[1]}_{band1}_{metric}_{event}_pair_level_df_intx')
    full_data_df.append(test_df)

full_data_df = pd.concat(full_data_df)


In [ ]:
full_data_df.reset_index(drop=True, inplace=True)
full_data_df['label'] = full_data_df.pair_label.apply(lambda x: x.split(':')[1])
full_data_df = full_data_df.groupby(['participant', 'label', 'roi1']).mean(numeric_only=True).reset_index()
                                    
# full_data_df = full_data_df.loc[full_data_df.groupby(['participant', 'label', 'roi1'])['phit'].idxmax()]
#.idxmax()]
#.apply(lambda x: x.abs().idxmax())]


In [ ]:
full_data_df[full_data_df.roi1=='OFC']

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band1}_coh_rpe_eleclevel.pdf') as pdf:
    sns.barplot(data=full_data_df, x='roi1', y='rpe',errorbar='se',
                alpha = 0.5, color='k')
    sns.stripplot(data=full_data_df[(np.abs(full_data_df['rpe'])<=1.96)], x='roi1', y='rpe',
                  alpha=0.1, color='k')
    sns.stripplot(data=full_data_df[(np.abs(full_data_df['rpe'])>1.96)], x='roi1', y='rpe',
                  alpha=0.5, color='red')

    plt.hlines([0], -1, 6, 'k')
    ax.set(ylabel='RPE:memory interaction', xlim=[-1,6], ylim=[-3.75, 3.75])
    pdf.savefig(fig)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band1}_coh_phit_eleclevel.pdf') as pdf:
    sns.barplot(data=full_data_df, x='roi1', y='phit',errorbar='se',
                alpha = 0.5, color='k')
    sns.stripplot(data=full_data_df[(np.abs(full_data_df['phit'])<=1.96)], x='roi1', y='phit',
                  alpha=0.1, color='k')
    sns.stripplot(data=full_data_df[(np.abs(full_data_df['phit'])>1.96)], x='roi1', y='phit',
                  alpha=0.5, color='red')

    plt.hlines([0], -1, 6, 'k')
    ax.set(ylabel='memory effect', xlim=[-1,6], ylim=[-3.75, 3.75])
    pdf.savefig(fig)

In [ ]:
gamble_model_df.rename(columns={'channel': 'label'}, inplace=True)

# roi = 'dlPFC'
roi_data = full_data_df.merge(gamble_model_df, on=['participant', 'label'])
roi_data.rename(columns={'phit:zrpe':'hfa_intx'}, inplace=True)

In [ ]:
sns.regplot(x=roi_data[roi_data.roi1=='HPC'].rpe_x, 
            y=roi_data[roi_data.roi1=='HPC'].phit_x)

In [ ]:
# # sns.lmplot(data=gamble_model_df[gamble_model_df.roi1=='dmPFC'], x='phit_y', y='hfa_intx', hue='participant', ci=None)
# # sns.regplot(data=gamble_model_df[gamble_model_df.roi1=='dmPFC'], x='phit_y', y='hfa_intx')

with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{roi}_OFC_coh_phit:rpe.pdf') as pdf:

    fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)

    # sns.lmplot(x='phit_x', y='hfa_intx', data=roi_data, scatter_kws={'alpha':0.3}, hue='participant', ci=None)

    sns.regplot(x='phit_x', y='hfa_intx', data=roi_data, scatter=None, color='k', ax=ax)
    for subj in roi_data.participant.unique():
        sns.regplot(x='phit_x', y='hfa_intx', data=roi_data[roi_data.participant==subj], 
                    ci=None,
                    line_kws={'alpha':0.3},
                    scatter_kws={'alpha':0.3}, 
                    color='brown', ax=ax)
        
    sns.despine()

    pdf.savefig()

In [ ]:
result = smf.mixedlm(formula = 'hfa_intx ~ 1 + phit_x',
                      re_formula='1',
                      groups='participant',
                      data=roi_data).fit(reml=False)

In [ ]:
result.summary()

In [ ]:
all_elec_dfs = []
# subj_ids = ['UI001', 'UI002', 'UI003', 'UI004']
for subj_id in subj_ids: 
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
    elec_df['participant'] = subj_id
    all_elec_dfs.append(elec_df)
all_elec_dfs = pd.concat(all_elec_dfs).reset_index()

merged_elec_df = all_elec_dfs.merge(full_data_df[gamble_model_df.region==region], on=['participant', 'label'])

merged_elec_df.hemisphere = merged_elec_df.hemisphere.str.lower()

In [ ]:
merged_elec_df[merged_elec_df.roi1=='dmPFC']

In [ ]:
model_df.phit.mean()

In [ ]:
# Significance: 
region = 'dlPFC'

model_df = full_data_df[full_data_df.roi1==f'{region}']


predictor = 'phit'

if predictor == 'phit:zrpe':
    # rename this for our model
    model_df.rename(columns={'phit:zrpe':'phit_rpe_intx'}, inplace=True)
    predictor = 'phit_rpe_intx'

model = bmb.Model(f'{predictor} ~ 1 + (1|participant)', 
                  model_df, 
                  family='gaussian', categorical=['participant'])

# Fit the model
results = model.fit(draws=1000, chains=4)

df = az.summary(results)

# if results.pvalues['Intercept']<=0.025:
if np.sign(df.loc['Intercept']['hdi_3%']) == np.sign(df.loc['Intercept']['hdi_97%']):
    
    # print(results.summary())
    with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band1}_coh_{region}_{predictor}_M_on_betas.pdf') as pdf:

        # Create a scatter plot of individual data points
        fig, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
        sns.stripplot(x="participant", y=f'{predictor}', data=model_df, 
                      jitter=True, alpha=0.6, color="gray", s=6, ax=ax)

        # plot the mean line
        subject_means = model_df.groupby("participant")[f'{predictor}'].mean()

        sns.boxplot(showmeans=True,
                    meanline=True,
                    meanprops={'color': 'k', 'ls': '-', 'lw': 2},
                    medianprops={'visible': False},
                    whiskerprops={'visible': False},
                    zorder=10,
                    x=subject_means.index,
                    y=subject_means.values,
                    showfliers=False,
                    showbox=False,
                    showcaps=False,
                    ax=ax)


        # # Overlay the subject means
        overall_mean = df.loc['Intercept']['mean']
        # Plot the overall mean from the mixed-effects model
        plt.axhline(overall_mean, color="blue", linestyle="--", label=f"Overall Mean: {overall_mean:.2f}")
        plt.axhline(df.loc['Intercept']['hdi_3%'], color='red', linestyle='--')
        plt.axhline(df.loc['Intercept']['hdi_97%'], color='red', linestyle='--')

        plt.axhline(0, color='black')
        sns.despine()
        
        pdf.savefig()

In [ ]:
df

In [ ]:
region = 'dmPFC'

model_df = merged_elec_df[merged_elec_df.roi1==f'{region}']
plot_df = model_df.groupby(['participant', 'label']).mean(numeric_only=True).reset_index().merge(test_df, on=['participant', 'label'])
sns.regplot(data=plot_df, x='phit_rpe_intx',
            y='half_q')


In [ ]:
        fig, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
        sns.stripplot(x="participant", y=f'{predictor}', data=full_data_df[full_data_df.paired_regions==region], 
                      jitter=True, alpha=0.6, color="gray", s=6, ax=ax)

        # plot the mean line
        subject_means = full_data_df[full_data_df.paired_regions==region].groupby("participant")[f'{predictor}'].mean()

        sns.boxplot(showmeans=True,
                    meanline=True,
                    meanprops={'color': 'k', 'ls': '-', 'lw': 2},
                    medianprops={'visible': False},
                    whiskerprops={'visible': False},
                    zorder=10,
                    x=subject_means.index,
                    y=subject_means.values,
                    showfliers=False,
                    showbox=False,
                    showcaps=False,
                    ax=ax)


        # # Overlay the subject means
        overall_mean = df.loc['Intercept']['mean']
        # Plot the overall mean from the mixed-effects model
        plt.axhline(overall_mean, color="blue", linestyle="--", label=f"Overall Mean: {overall_mean:.2f}")
        plt.axhline(df.loc['Intercept']['hdi_3%'], color='red', linestyle='--')
        plt.axhline(df.loc['Intercept']['hdi_97%'], color='red', linestyle='--')

        plt.axhline(0, color='black')
        sns.despine()

In [ ]:
df

9/29/2024: Run the following to correlate an electrode's theta connectivity with a region to it's integration of RPE and memory in HFA

In [ ]:
metric = 'coh'
band1 = 'theta'
band2 = 'theta'
event = 'feedback_start' 
savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

pair = ['dmPFC', 'OFC']

all_pairs_df = pd.read_csv(f'{savepath}/{pair[0]}_{pair[1]}_{metric}_{band1}_{band2}_{event}_df')


In [ ]:
all_pairs_df['label'] = np.nan
all_pairs_df['label'] = all_pairs_df.pair_label.apply(lambda x: x.split(':')[1])

In [ ]:
test_wpli_df = all_pairs_df.groupby(['participant', 'label']).mean(numeric_only=True).reset_index()

In [ ]:
res_df = gamble_model_df[gamble_model_df.region=='OFC']
res_df.rename(columns={'channel':'label'}, inplace=True)

In [ ]:
# merging here is wrong for coh and pac

In [ ]:
merged_df = res_df.merge(test_wpli_df, on=['participant', 'label'])
merged_df.rename(columns={'phit:zrpe': 'phit_rpe_intx'}, inplace=True)

In [ ]:
merged_df

In [ ]:
    sns.lmplot(data=merged_df, x='conn', y='phit_rpe_intx', hue='participant', ci=None)
    sns.regplot(data=merged_df, x='conn', y='phit_rpe_intx', color='k', ci=95, scatter=None)

In [ ]:
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{pair[0]}_coh_phit:rpe.pdf') as pdf:

    sns.lmplot(data=merged_df, x='conn', y='phit_rpe_intx', hue='participant', ci=None, 
               palette=sns.color_palette(['purple'], len(merged_df['participant'].unique())), 
               scatter_kws={'alpha':0.2}, line_kws={'alpha':0.2}, height=5, legend=None)
    sns.regplot(data=merged_df, x='conn', y='phit_rpe_intx', color='k', ci=95, scatter=None)

    plt.ylim([-4, 4])
    
    pdf.savefig()

In [ ]:
results = smf.mixedlm('phit_rpe_intx ~ 1 + conn', data=merged_df, 
                      re_formula = '1', 
                      groups='participant').fit()

In [ ]:
results.summary()

## Check the relationship between OFC rpe:phit and OFC beta hfa PAC

In [ ]:
roi = 'OFC'
metric = 'pac'
event = 'feedback_start' 
save_path = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'
all_elec_df = pd.read_csv(f'{save_path}/all_{roi}_pac_{event}.csv')

In [ ]:
band = 'hfa'
analysis_evs = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')


res_df = gamble_model_df[gamble_model_df.region=='OFC']
res_df.rename(columns={'channel':'label'}, inplace=True)

In [ ]:
res_df = gamble_model_df[gamble_model_df.region=='OFC']
res_df.rename(columns={'channel':'label'}, inplace=True)

In [ ]:
test_pac_df = all_elec_df.groupby(['participant', 'label']).mean(numeric_only=True).reset_index()

In [ ]:
all_elec_df

In [ ]:
merged_df = res_df.merge(test_pac_df, on=['participant', 'label'])
merged_df.rename(columns={'phit:zrpe': 'phit_rpe_intx'}, inplace=True)

In [ ]:
    sns.lmplot(data=merged_df, x='avg_conn', y='phit_rpe_intx', hue='participant', ci=None)
    sns.regplot(data=merged_df, x='avg_conn', y='phit_rpe_intx', color='k', ci=95, scatter=None)

In [ ]:
results = smf.mixedlm('phit_rpe_intx ~ 1 + avg_conn', data=merged_df, 
                      re_formula = '1 + avg_conn', 
                      groups='participant').fit()

In [ ]:
results.summary()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5), dpi=300)

sns.barplot(data=full_data_df, x='paired_regions', y='phit:zrpe',errorbar='se', alpha = 0.5, color='k')
sns.stripplot(data=full_data_df[np.abs(full_data_df['phit:zrpe'])<=1.96], x='paired_regions', y='phit:zrpe', alpha=0.1, color='k')
sns.stripplot(data=full_data_df[np.abs(full_data_df['phit:zrpe'])>1.96], x='paired_regions', y='phit:zrpe', alpha=0.5, color='red')

sns.despine()

In [ ]:
pair_region = 'HPC_OFC'
predictor = 'phit:zrpe'

if predictor == 'phit:zrpe':
    # rename this for our model
    full_data_df.rename(columns={'phit:zrpe':'phit_rpe_intx'}, inplace=True)
    predictor = 'phit_rpe_intx'

model = bmb.Model(f'{predictor} ~ 1 + (1|participant)', 
                  full_data_df[full_data_df.paired_regions==pair_region], 
                  family='gaussian', categorical=['participant'])

# Fit the model
results = model.fit(draws=1000, chains=4)

df = az.summary(results)

# if results.pvalues['Intercept']<=0.025:
if np.sign(df.loc['Intercept']['hdi_3%']) == np.sign(df.loc['Intercept']['hdi_97%']):
    
    # # print(results.summary())
    # with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_{region}_{predictor}_M_on_betas.pdf') as pdf:

    # Create a scatter plot of individual data points
    fig, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
    sns.stripplot(x="participant", y=f'{predictor}', data=full_data_df[full_data_df.paired_regions==pair_region], 
                  jitter=True, alpha=0.6, color="gray", s=6, ax=ax)

    # plot the mean line
    subject_means = full_data_df[full_data_df.paired_regions==pair_region].groupby("participant")[f'{predictor}'].mean()

    sns.boxplot(showmeans=True,
                meanline=True,
                meanprops={'color': 'k', 'ls': '-', 'lw': 2},
                medianprops={'visible': False},
                whiskerprops={'visible': False},
                zorder=10,
                x=subject_means.index,
                y=subject_means.values,
                showfliers=False,
                showbox=False,
                showcaps=False,
                ax=ax)


    # # Overlay the subject means
    overall_mean = df.loc['Intercept']['mean']
    # Plot the overall mean from the mixed-effects model
    plt.axhline(overall_mean, color="blue", linestyle="--", label=f"Overall Mean: {overall_mean:.2f}")
    plt.axhline(df.loc['Intercept']['hdi_3%'], color='red', linestyle='--')
    plt.axhline(df.loc['Intercept']['hdi_97%'], color='red', linestyle='--')

    plt.axhline(0, color='black')
    sns.despine()
        
        # pdf.savefig()

In [ ]:
df

## Timewise directed connectivity analysis of theta connectivity (tr-GC)


In [ ]:
metric = 'granger' 

savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'
source_region = 'dmPFC' 
target_region = 'OFC' 

band1 = 'theta' 
band2 = 'theta' 
event = 'feedback_start' 

conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z.npy')
freqs = np.linspace(2, 8, conn_group_data_neg[:, :].shape[1])


meta_data_neg = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z_df')
meta_data_neg['conn'] = np.nanmean(conn_group_data_neg[:, :], axis=1)


conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_Z.npy')
meta_data_pos = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_Z_df')
meta_data_pos['conn'] = np.nanmean(conn_group_data_pos[:, :], axis=1)

meta_data1 = pd.concat([meta_data_neg, meta_data_pos]) 

source_region = 'AMY' 
target_region = 'OFC' 

band1 = 'theta' 
band2 = 'theta' 
event = 'feedback_start' 

conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z.npy')
meta_data_neg = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z_df')
meta_data_neg['conn'] = np.nanmean(conn_group_data_neg[:, :], axis=1)

conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_Z.npy')
meta_data_pos = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_Z_df')
meta_data_pos['conn'] = np.nanmean(conn_group_data_pos[:, :], axis=1)

meta_data2 = pd.concat([meta_data_neg, meta_data_pos]) 

source_region = 'dlPFC' 
target_region = 'OFC' 

band1 = 'theta' 
band2 = 'theta' 
event = 'feedback_start' 

conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z.npy')
meta_data_neg = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_miss_Z_df')
meta_data_neg['conn'] = np.nanmean(conn_group_data_neg[:, :], axis=1)

conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_Z.npy')
meta_data_pos = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_hit_Z_df')
meta_data_pos['conn'] = np.nanmean(conn_group_data_pos[:, :], axis=1)

meta_data3 = pd.concat([meta_data_neg, meta_data_pos]) 

# meta_data = meta_data1.merge([meta_data2, meta_data3])

# predictor = 'conn'
# model = bmb.Model(f'{predictor} ~ 1 + cond + (1|participant)', 
#                   meta_data, 
#                   family='gaussian', categorical=['cond', 'participant'])

# # Fit the model
# results = model.fit(draws=1000, chains=4, target_accept=0.9)

# df = az.summary(results)

# # if results.pvalues['Intercept']<=0.025:
# if np.sign(df.loc['cond']['hdi_3%']) == np.sign(df.loc['cond']['hdi_97%']):
    
#     # print(results.summary())
#     # with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_{region}_{predictor}_M_on_betas.pdf') as pdf:

#     # Create a scatter plot of individual data points
#     fig, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
#     sns.stripplot(x="participant", y=f'{predictor}', data=meta_data, 
#                   jitter=True, alpha=0.6, color="gray", s=6, ax=ax)

#     # plot the mean line
#     subject_means = meta_data.groupby("participant")[f'{predictor}'].mean()

#     sns.boxplot(showmeans=True,
#                 meanline=True,
#                 meanprops={'color': 'k', 'ls': '-', 'lw': 2},
#                 medianprops={'visible': False},
#                 whiskerprops={'visible': False},
#                 zorder=10,
#                 x=subject_means.index,
#                 y=subject_means.values,
#                 showfliers=False,
#                 showbox=False,
#                 showcaps=False,
#                 ax=ax)


#     # # Overlay the subject means

#     # plt.scatter(subject_means.index, subject_means.values, color="red", label="Subject Means", zorder=3)

#     # Plot the overall mean from the mixed-effects model
#     # overall_mean = results.fe_params['Intercept']
#     plt.axhline(df.loc['Intercept']['mean'], color="blue", linestyle="--", label=f"Overall Mean: {df.loc['Intercept']['mean']:.2f}")
#     plt.axhline(df.loc['Intercept']['hdi_3%'], color='red', linestyle='--')
#     plt.axhline(df.loc['Intercept']['hdi_97%'], color='red', linestyle='--')

#     plt.axhline(0, color='black')
#     sns.despine()
        
#         # pdf.savefig()

In [ ]:

meta_data1['label'] = np.nan
meta_data1['label'] = meta_data1.pair_labels.apply(lambda x: x.split(':')[1])


meta_data2['label'] = np.nan
meta_data2['label'] = meta_data2.pair_labels.apply(lambda x: x.split(':')[1])

meta_data3['label'] = np.nan
meta_data3['label'] = meta_data3.pair_labels.apply(lambda x: x.split(':')[1])

In [ ]:
meta_data1

In [ ]:
plt.hist(meta_data1[(meta_data1.participant=='MS012') & (meta_data1.label=='rmolf1-rmolf2')].conn)

In [ ]:
test_df3 = meta_data3.groupby(['participant', 'label', 'cond']).mean(numeric_only=True).reset_index()

In [ ]:
test_df3  = test_df3[test_df3.cond=='miss']

# test_df1['half_q'] = half_quotient

In [ ]:
test_df3.merge(test_df1, on=['participant', 'label'])

In [ ]:
sns.regplot(data=test_df1, x='conn', y='half_q')

In [ ]:
sns.regplot(data=test_df3.merge(test_df1, on=['participant', 'label']), 
           x='conn_x', y='half_q')

none of the theta granger causalities predict whether an electrode will be an early or late rpe:phit responder! 



In [ ]:
merged_df = meta_data1.merge(meta_data2, on=['participant', 'label', 'cond'])
merged_df['conn_x'] = -merged_df['conn_x']

In [ ]:
merged_df['combined_conn'] = merged_df['conn_x'] * merged_df['conn_y']

In [ ]:
test_df = merged_df.groupby(['participant', 'label', 'cond']).mean(numeric_only=True).reset_index()

In [ ]:
test_df2 = res_df.merge(test_df, on=['participant', 'label'])

In [ ]:
    sns.lmplot(data=test_df2[test_df2.cond=='miss'], x='phit:zrpe', y='combined_conn', hue='participant', ci=None, 
               palette=sns.color_palette(['purple'], len(merged_df['participant'].unique())), 
               scatter_kws={'alpha':0.2}, line_kws={'alpha':0.2}, height=5, legend=None)
    sns.regplot(data=test_df2[test_df2.cond=='miss'], x='phit:zrpe', y='combined_conn', color='k', ci=95, scatter=None)

    plt.ylim([-1, 1])

In [ ]:
band = 'hfa'
analysis_evs = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')


res_df = gamble_model_df[gamble_model_df.region=='OFC']
res_df.rename(columns={'channel':'label'}, inplace=True)

In [ ]:
merged_df = res_df.merge(meta_data1, on=['participant', 'label'])

# merged_df = res_df.merge(meta_data2, on=['participant', 'label'])

# merged_df = res_df.merge(meta_data3, on=['participant', 'label'])


merged_df.rename(columns={'phit:zrpe': 'phit_rpe_intx'}, inplace=True)


In [ ]:
merged_df = merged_df.groupby(['participant', 'label', 'cond']).mean(numeric_only=True).reset_index()

In [ ]:
sns.pointplot(data=merged_df, x='cond', y='conn', order=['miss', 'hit'])

In [ ]:
sns.lmplot(data=merged_df[merged_df.cond=='hit'], x='conn', y='phit_rpe_intx', hue='participant', ci=None, 
           palette=sns.color_palette(['purple'], len(merged_df['participant'].unique())), 
           scatter_kws={'alpha':0.2}, line_kws={'alpha':0.2}, height=5, legend=None)
sns.regplot(data=merged_df[merged_df.cond=='hit'], x='conn', y='phit_rpe_intx', color='k', ci=95, scatter=None)

plt.ylim([-4, 4])

In [ ]:
merged_df

In [ ]:
results = smf.mixedlm('phit_rpe_intx ~ 1 + conn', data=merged_df[merged_df.cond=='hit'], 
                      re_formula = '1', 
                      groups='participant').fit()

In [ ]:
results.summary()

## Main trialwise directed connectivity analysis of theta connectivity (tr-GC)

In [ ]:
pairs = [['OFC', 'OFC']
        ]

metric = 'coh'
band1 = 'theta'
band2 = 'theta'
event = 'feedback_start' 
savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

for pair in pairs:
    
    all_pairs_df = pd.read_csv(f'{savepath}/{pair[0]}_{pair[1]}_{metric}_{band1}_{band2}_{event}_df')

    # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 
    
    subj_list = all_pairs_df[['participant', 'pair_label']].drop_duplicates()['participant'].values
    pair_list = all_pairs_df[['participant', 'pair_label']].drop_duplicates()['pair_label'].values

    model_results = {'participant':[], 'pair_label':[], 'roi1':[], 'roi2':[], 'unique_pair_id':[], 
                     'rpe':[], 'pm':[], 'phit':[], 'phit:zrpe':[]} 
    pair_counter = 0
    
    for ix, _ in enumerate(pair_list):
        conn_df = all_pairs_df[(all_pairs_df.participant==subj_list[ix]) & (all_pairs_df.pair_label==pair_list[ix])]

        # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 

        if len(conn_df)>80:
            conn_df = conn_df.iloc[0:80]

        conn_df['unique_pair_id'] = pair_counter
        conn_df['zrpe'] = (conn_df.rpe - np.nanmean(conn_df.rpe)) / (2*np.nanstd(conn_df.rpe))
        conn_df['urpe'] = np.abs(conn_df.rpe)
        conn_df['zrpe_unsigned'] = (conn_df.urpe - np.nanmean(conn_df.urpe)) / (2*np.nanstd(conn_df.urpe))
        conn_df['zpm'] = (conn_df.DPRIME - np.nanmean(conn_df.DPRIME)) / (2*np.nanstd(conn_df.DPRIME))

        model_results['participant'].append(conn_df.participant.unique()[0])
        model_results['pair_label'].append(conn_df.pair_label.unique()[0])
        model_results['roi1'].append(conn_df.roi1.unique()[0])
        model_results['roi2'].append(conn_df.roi2.unique()[0])
        model_results['unique_pair_id'].append(conn_df.unique_pair_id.unique()[0])

        formula = 'conn ~ 1 + zrpe + zpm'
        results = statistics_utils.permutation_regression_zscore(conn_df, 
                                                formula, 
                                                n_permutations=1000, 
                                                plot_res=False)

        model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
        model_results['pm'].append(results.loc['zpm']['Z_Score'])

        memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_list[ix])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
        memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
        memory_df['phit'] = np.where(memory_df.hits == True, 0.5, -0.5)
        new_df = conn_df.merge(memory_df, on=['participant', 'trials'])

        formula = 'conn ~ 1 + zrpe*phit'

        results = statistics_utils.permutation_regression_zscore(new_df, 
                                                                 'conn ~ 1 + C(phit)*zrpe', n_permutations=1000, plot_res=False)

        model_results['phit'].append(results.loc['C(phit)[T.0.5]']['Z_Score'])
        model_results['phit:zrpe'].append(results.loc['C(phit)[T.0.5]:zrpe']['Z_Score'])

        pair_counter += 1

    gamble_model_df = pd.DataFrame(model_results)
    gamble_model_df['paired_regions'] = f'{pair[0]}_{pair[1]}'
    gamble_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{pair[0]}_{pair[1]}_{band1}_{metric}_{event}_pair_level_df_intx', index=False)


In [ ]:
gamble_model_df['phit:zrpe'].mean()

In [ ]:
sns.barplot(data=gamble_model_df, x='paired_regions', y='phit:zrpe',errorbar='se', alpha = 0.5, color='k')
sns.stripplot(data=gamble_model_df[np.abs(gamble_model_df['phit:zrpe'])<=1.96], x='paired_regions', y='phit:zrpe', alpha=0.1, color='k')
sns.stripplot(data=gamble_model_df[np.abs(gamble_model_df['phit:zrpe'])>1.96], x='paired_regions', y='phit:zrpe', alpha=0.5, color='red')


In [ ]:
    all_pairs_df = pd.read_csv(f'{savepath}/{pair[0]}_{pair[1]}_{metric}_{band1}_{band2}_{event}_df')


In [ ]:
all_pairs_df

In [ ]:
pairs = [['AMY', 'OFC'], 
        ['HPC', 'OFC'], 
        ['dmPFC', 'OFC'],
         ['dlPFC', 'OFC'],
         ['OFC', 'OFC']
        ]

full_data_df = []
for pair in pairs:
    
    gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{pair[0]}_{pair[1]}_{band1}_{metric}_{event}_pair_level_df_intx')
    full_data_df.append(gamble_model_df)

full_data_df = pd.concat(full_data_df)


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5), dpi=300)

sns.barplot(data=full_data_df, x='paired_regions', y='phit:zrpe',errorbar='se', alpha = 0.5, color='k')
sns.stripplot(data=full_data_df[np.abs(full_data_df['phit:zrpe'])<=1.96], x='paired_regions', y='phit:zrpe', alpha=0.1, color='k')
sns.stripplot(data=full_data_df[np.abs(full_data_df['phit:zrpe'])>1.96], x='paired_regions', y='phit:zrpe', alpha=0.5, color='red')

sns.despine()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5), dpi=300)

sns.barplot(data=full_data_df, x='paired_regions', y='rpe',errorbar='se', alpha = 0.5, color='k')
sns.stripplot(data=full_data_df[np.abs(full_data_df['rpe'])<=1.96], x='paired_regions', y='rpe', alpha=0.1, color='k')
sns.stripplot(data=full_data_df[np.abs(full_data_df['rpe'])>1.96], x='paired_regions', y='rpe', alpha=0.5, color='red')

sns.despine()

In [ ]:
pair_region = 'AMY_OFC'
predictor = 'phit:zrpe'

if predictor == 'phit:zrpe':
    # rename this for our model
    full_data_df.rename(columns={'phit:zrpe':'phit_rpe_intx'}, inplace=True)
    predictor = 'phit_rpe_intx'

model = bmb.Model(f'{predictor} ~ 1 + (1|participant)', 
                  full_data_df[full_data_df.paired_regions==pair_region], 
                  family='gaussian', categorical=['participant'])

# Fit the model
results = model.fit(draws=1000, chains=4)

df = az.summary(results)

# if results.pvalues['Intercept']<=0.025:
if np.sign(df.loc['Intercept']['hdi_3%']) == np.sign(df.loc['Intercept']['hdi_97%']):
    
    # # print(results.summary())
    # with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_{region}_{predictor}_M_on_betas.pdf') as pdf:

    # Create a scatter plot of individual data points
    fig, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
    sns.stripplot(x="participant", y=f'{predictor}', data=full_data_df[full_data_df.paired_regions==pair_region], 
                  jitter=True, alpha=0.6, color="gray", s=6, ax=ax)

    # plot the mean line
    subject_means = full_data_df[full_data_df.paired_regions==pair_region].groupby("participant")[f'{predictor}'].mean()

    sns.boxplot(showmeans=True,
                meanline=True,
                meanprops={'color': 'k', 'ls': '-', 'lw': 2},
                medianprops={'visible': False},
                whiskerprops={'visible': False},
                zorder=10,
                x=subject_means.index,
                y=subject_means.values,
                showfliers=False,
                showbox=False,
                showcaps=False,
                ax=ax)


    # # Overlay the subject means
    overall_mean = df.loc['Intercept']['mean']
    # Plot the overall mean from the mixed-effects model
    plt.axhline(overall_mean, color="blue", linestyle="--", label=f"Overall Mean: {overall_mean:.2f}")
    plt.axhline(df.loc['Intercept']['hdi_3%'], color='red', linestyle='--')
    plt.axhline(df.loc['Intercept']['hdi_97%'], color='red', linestyle='--')

    plt.axhline(0, color='black')
    sns.despine()
        
        # pdf.savefig()

In [ ]:
df

## When does OFC communicate between the relevant elecs for encoding and the relevant elecs for retrieval?

In [ ]:
OFC_coh_encoding = pd.read_csv('/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/coh/OFC_OFC_coh_theta_theta_feedback_start_df')
# dmPFC_coh_encoding = pd.read_csv('/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/coh/dlPFC_OFC_coh_theta_theta_feedback_start_df')


# OFC_coh_cue = pd.read_csv('/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/coh/OFC_OFC_coh_theta_theta_cue_on_df')


In [ ]:
# OFC_coh_cue['trials'] = OFC_coh_encoding['trials']


In [ ]:
memory_df = combined_df[(combined_df.condition == 'Day1')].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
memory_df['trials'] = memory_df['trials_gamble']

In [ ]:
OFC_coh_encoding = OFC_coh_encoding.merge(memory_df, on=['participant', 'trials'])

In [ ]:
# Compute an SME for the coherence between each pair. Higher values = more ocherence for hits than misses 
results = []

# Group by participant and pair_label
for (participant, pair_label), group in OFC_coh_encoding.groupby(['participant', 'pair_label']):
    conn_true = group[group['hits'] == True]['conn']
    conn_false = group[group['hits'] == False]['conn']
    
    if len(conn_true) > 0 and len(conn_false) > 0:
        # Perform Mann-Whitney U test
        # stat, p_value = mannwhitneyu(conn_true, conn_false, alternative='two-sided')
        stat, p = ttest_ind(conn_true, conn_false)
        sme = stat  # Statistic can be used as a summary measure of effect (sme)
        
        results.append({
            'participant': participant,
            'pair_label': pair_label,
            'sme': sme
        })

# Convert results to a dataframe
results_df = pd.DataFrame(results)
# print(results_df)

In [ ]:
# test_df1 =OFC_coh_encoding.groupby(['participant', 'pair_label']).mean(numeric_only=True).reset_index()
# test_df1['label'] = test_df1.pair_label.apply(lambda x: x.split(':')[1])
# test_df1 = test_df1.groupby(['participant', 'label']).mean(numeric_only=True).reset_index()

# test_df2 = dmPFC_coh_encoding.groupby(['participant', 'pair_label']).mean(numeric_only=True).reset_index()
# test_df2['label'] = test_df2.pair_label.apply(lambda x: x.split(':')[1])
# test_df2 = test_df2.groupby(['participant', 'label']).mean(numeric_only=True).reset_index()


In [ ]:
# sns.regplot(data=test_df2.merge(test_df1, on=['participant', 'label']), x='conn_x', y='conn_y')

In [ ]:
# ttest_rel(OFC_coh_encoding.groupby(['participant', 'pair_label']).mean(numeric_only=True).reset_index().conn, 
#           OFC_coh_cue.groupby(['participant', 'pair_label']).mean(numeric_only=True).reset_index().conn) 
          

In [ ]:
# df = OFC_coh_encoding.groupby(['participant', 'pair_label']).mean(numeric_only=True).reset_index()
# df = df[['participant', 'pair_label', 'conn']]
df = results_df
df[['electrode1', 'electrode2']] = df['pair_label'].str.split(':', expand=True)


band = 'hfa'
analysis_evs = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')
gamble_model_df = gamble_model_df[['participant', 'channel', 'phit:zrpe']]

cue_model_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_multinomial_df_wrpe') 
cue_model_df = cue_model_df[cue_model_df.category=='miss']
cue_model_df = cue_model_df[['participant', 'label', 'hit_compared_to_coefficient']]
cue_model_df.rename(columns={'hit_compared_to_coefficient':'phit:zrpe'}, inplace=True)

# Merge 'phit:zrpe' from cue_model_df for 'electrode1' (this will have a different column name)
df = pd.merge(df, cue_model_df, left_on=['participant', 'electrode1'], right_on=['participant', 'label'], how='left')
df = df.rename(columns={'phit:zrpe': 'cue_phit:zrpe_electrode1'})
# df = df.drop(columns=['channel'])  # Drop the 'channel' column as it's redundant

# Merge 'phit:zrpe' from cue_model_df for 'electrode2' (again with a different column name)
df = pd.merge(df, cue_model_df, left_on=['participant', 'electrode2'], right_on=['participant', 'label'], how='left')
df = df.rename(columns={'phit:zrpe': 'cue_phit:zrpe_electrode2'})
# df = df.drop(columns=['channel'])  # Drop the 'channel' column again

# Merge 'phit:zrpe' from gamble_model_df for 'electrode1'
df = pd.merge(df, gamble_model_df, left_on=['participant', 'electrode1'], right_on=['participant', 'channel'], how='left')
df = df.rename(columns={'phit:zrpe': 'phit:zrpe_electrode1'})
df = df.drop(columns=['channel'])  # Drop the 'channel' column as it's redundant after merge

# Merge 'phit:zrpe' from gamble_model_df for 'electrode2'
df = pd.merge(df, gamble_model_df, left_on=['participant', 'electrode2'], right_on=['participant', 'channel'], how='left')
df = df.rename(columns={'phit:zrpe': 'phit:zrpe_electrode2'})
df = df.drop(columns=['channel'])  # Drop the 'channel' column again

In [ ]:
# Sort the dataframe by 'phit:zrpe_electrode1' for the x-axis and 'cue_phit:zrpe_electrode2' for the y-axis
df = df.sort_values(by=['phit:zrpe_electrode1', 'cue_phit:zrpe_electrode2'])

# Create unique electrode IDs by combining participant and electrode labels
df['unique_electrode1'] = df['participant'] + '_' + df['electrode1']
df['unique_electrode2'] = df['participant'] + '_' + df['electrode2']

# Create a pivot table for the heatmap (using unique electrode labels)
heatmap_data = df.pivot(index='unique_electrode2', columns='unique_electrode1', values='sme')

# Plot the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, cmap='coolwarm', cbar_kws={'label': 'Conn'})
plt.title('Electrode Pair Connectivity Heatmap (Sorted by participant)')
plt.show()

In [ ]:
# Sort the dataframe by 'phit:zrpe_electrode1' for the x-axis and 'cue_phit:zrpe_electrode2' for the y-axis
df = df.sort_values(by=['phit:zrpe_electrode1', 'cue_phit:zrpe_electrode2'])

# Create unique electrode IDs by combining participant and electrode labels
df['unique_electrode1'] = df['participant'] + '_' + df['electrode1']
df['unique_electrode2'] = df['participant'] + '_' + df['electrode2']

# Create a pivot table for the heatmap (using unique electrode labels)
heatmap_data = df.pivot(index='unique_electrode2', columns='unique_electrode1', values='sme')

# Reorder the columns (unique_electrode1) by the sorted 'phit:zrpe_electrode1' values
heatmap_data = heatmap_data.reindex(sorted(df['unique_electrode1'].unique(), key=lambda x: df[df['unique_electrode1'] == x]['phit:zrpe_electrode1'].values[0]), axis=1)

# Reorder the rows (unique_electrode2) by the sorted 'cue_phit:zrpe_electrode2' values
heatmap_data = heatmap_data.reindex(sorted(df['unique_electrode2'].unique(), key=lambda x: df[df['unique_electrode2'] == x]['cue_phit:zrpe_electrode2'].values[0]), axis=0)

# Plot the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, cmap='coolwarm', cbar_kws={'label': 'Conn'})
plt.title('Electrode Pair Connectivity Heatmap (Sorted by phit:zrpe_electrode1 and cue_phit:zrpe_electrode2)')
plt.show()

In [ ]:
# Get the shape of the heatmap_data
num_rows, num_cols = heatmap_data.shape

# Determine midpoints
mid_row = num_rows // 2
mid_col = num_cols // 2


# Calculate the cornermost half dimensions for each quadrant
top_left_quadrant = heatmap_data.iloc[:mid_row, :mid_col].dropna(how='all')
top_right_quadrant = heatmap_data.iloc[:mid_row, mid_col:].dropna(how='all')
bottom_left_quadrant = heatmap_data.iloc[mid_row:, :mid_col].dropna(how='all')
bottom_right_quadrant = heatmap_data.iloc[mid_row:, mid_col:].dropna(how='all')

# Determine the number of rows and columns for cornermost half
corner_rows = top_left_quadrant.shape[0] // 2
corner_cols = top_left_quadrant.shape[1] // 2

# Extract the cornermost half of each quadrant
top_left_corner = top_left_quadrant.iloc[:corner_rows, :corner_cols]
top_right_corner = top_right_quadrant.iloc[:corner_rows, :corner_cols]
bottom_left_corner = bottom_left_quadrant.iloc[-corner_rows:, :corner_cols]
bottom_right_corner = bottom_right_quadrant.iloc[-corner_rows:, :corner_cols]

# Remove NaN entries and stack each quadrant corner
top_left_values = top_left_corner.stack().reset_index(name='value')
top_right_values = top_right_corner.stack().reset_index(name='value')
bottom_left_values = bottom_left_corner.stack().reset_index(name='value')
bottom_right_values = bottom_right_corner.stack().reset_index(name='value')


In [ ]:
sns.distplot(top_right_values.value, color='k')
sns.distplot(bottom_right_values.value)

So, the electrodes that are encoding rpe:memory, and encoding recognition, are not more theta coherent than the electrodes that encode neither. During feedback_start, or during cue presentation.

In [ ]:
from scipy.stats import ttest_ind, ttest_rel, mannwhitneyu
mannwhitneyu(top_right_values.value, 
         top_left_values.value)
          

# Model single-electrode HFA during gamble for previous trial's RPE/PM/PHIT

In [ ]:
rois = ['dmPFC', 'OFC', 'HPC', 'AMY']

analysis_evs = 'gamble_start'

model_results = {'participant':[], 'channel':[], 'region':[], 'unique_elec_id':[], 
                 'rpe':[], 'pm':[], 'phit':[]} 
bands = ['hfa', 'theta']

for band in bands: 
    
    # RELOAD the data each time the model is run to ensure no funny business with dataframe manipulation between loop runs
    chan_dfs = [] 
    glms = []
    elec_counter = 0 
    for subj_id in subj_ids:
        print(f'Loading subj {subj_id}')
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        for chan in elec_df.label.unique():
            if elec_df[elec_df.label==chan].salman_region.values[0] in rois:
                chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time-1-0.csv')
                chan_df['unique_elec_id'] = elec_counter
                chan_df = chan_df[['participant', 'region', 'channel', 'unique_elec_id', 'trials', 'fband', 'tfr']]
                
                chan_df = chan_df.merge(learn_df, on=['participant', 'trials'])
                
                chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
                chan_df['urpe'] = np.abs(chan_df.rpe)
                chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
                chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
                
                # Shift to reflect that power on trial t+1 is related to variables on trial t
                chan_df['tfr'] = chan_df['tfr'].shift(len(chan_df.fband.unique()))
                
                model_results['participant'].append(chan_df.participant.unique()[0])
                model_results['channel'].append(chan_df.channel.unique()[0])
                model_results['region'].append(chan_df.region.unique()[0])
                model_results['unique_elec_id'].append(chan_df.unique_elec_id.unique()[0])
                
                formula = 'tfr ~ 1 + zrpe + zpm'
                results = permutation_regression_zscore(chan_df[chan_df.fband==band], 
                                                        formula, 
                                                        n_permutations=1000, 
                                                        plot_res=False)
                
                model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
                model_results['pm'].append(results.loc['zpm']['Z_Score'])
                
                memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
                memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
                memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
                chan_df = chan_df.merge(memory_df, on=['participant', 'trials'])
                
                formula = 'tfr ~ 1 + zrpe*phit'
                results = permutation_regression_zscore(chan_df[chan_df.fband==band], 
                                                        formula, 
                                                        n_permutations=1000, 
                                                        plot_res=False)
                
                model_results['phit'].append(results.loc['zrpe:phit']['Z_Score'])
                elec_counter +=1

    gamble_model_df = pd.DataFrame(model_results)
    gamble_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_rpe_pm_phit_eleclevel_df', index=False)


In [ ]:
band = 'theta'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_rpe_pm_phit_eleclevel_df')

# gamble_model_df = gamble_model_df.iloc[-450:]
gamble_model_df = gamble_model_df.iloc[450:900]

In [ ]:
gamble_model_df

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{analysis_evs}_{band}_rpe_eleclevel.pdf') as pdf:
    sns.barplot(data=gamble_model_df, x='region', y='rpe',errorbar='se', 
            order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=gamble_model_df[np.abs(gamble_model_df.rpe)<=1.96], x='region', y='rpe', 
        order = ['dmPFC', 'HPC','OFC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=gamble_model_df[np.abs(gamble_model_df.rpe)>1.96], x='region', y='rpe', 
        order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.5, color='red')

    plt.hlines([0], -1, 4, 'k')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,4], ylim=[-3.5, 3.5])
    pdf.savefig(fig)

power on the subsequent decision-making deliberation is not dependent on the rpe or memory of the previous trial

In [ ]:
# Are the region-grouped beta values different from 0?
p_values = [] 
for region in ['dmPFC', 'HPC', 'OFC', 'AMY']: 
    p_values.append(ttest_1samp(gamble_model_df[gamble_model_df.region==region].rpe.values, popmean=0)[1])
    

[x*4 for x in p_values]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{analysis_evs}_{band}_phit_eleclevel.pdf') as pdf:
    sns.barplot(data=gamble_model_df, x='region', y='phit',errorbar='se', 
            order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=gamble_model_df, x='region', y='phit', 
        order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=gamble_model_df[np.abs(gamble_model_df.phit)>1.96], x='region', y='phit', 
        order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.5, color='red')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='Hit sensitivity', xlim=[-1,4], ylim=[-3.5, 3.5])
    pdf.savefig(fig)

### Model single-electrode HFA during cue_on

9/9/2024: There is no relationship between RPE:Phit during the cue-on period. Our only hope here is connectivity (later this week)

In [ ]:
rois = ['dmPFC', 'OFC', 'HPC', 'AMY']

analysis_evs = 'cue_on'

In [ ]:
bands = ['theta', 'alpha', 'beta', 'hfa']
block = [0, 0.5]
for band in bands: 
    
    model_results = {'participant':[], 'channel':[], 'region':[], 'unique_elec_id':[], 
                 'rpe':[], 'pm':[], 'phit':[], 'phit:rpe':[]} 
    
    # RELOAD the data each time the model is run to ensure no funny business with dataframe manipulation between loop runs
    chan_dfs = [] 
    glms = []
    elec_counter = 0 
    for subj_id in subj_ids:
        print(f'Loading subj {subj_id}')
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        for chan in elec_df.label.unique():
            if elec_df[elec_df.label==chan].salman_region.values[0] in rois:
                chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time{block[0]}-{block[1]}.csv')
                chan_df['unique_elec_id'] = elec_counter
                chan_df = chan_df[['participant', 'region', 'channel', 'unique_elec_id', 'trials_gamble', 'fband', 'tfr']]
                chan_df.rename(columns={'trials_gamble': 'trials'}, inplace=True)
                chan_df = chan_df.merge(learn_df, on=['participant', 'trials'])
                
                chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
                chan_df['urpe'] = np.abs(chan_df.rpe)
                chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
                chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
                            
                model_results['participant'].append(chan_df.participant.unique()[0])
                model_results['channel'].append(chan_df.channel.unique()[0])
                model_results['region'].append(chan_df.region.unique()[0])
                model_results['unique_elec_id'].append(chan_df.unique_elec_id.unique()[0])
                
                memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
                memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
                memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
                chan_df = chan_df.merge(memory_df, on=['participant', 'trials'])
                
                formula = 'tfr ~ 1 + zrpe + zpm + C(phit) + C(phit):zrpe'
                
                results = permutation_regression_zscore(chan_df[chan_df.fband==band], 
                                                        formula, 
                                                        n_permutations=1000, 
                                                        plot_res=False)
                
                model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
                model_results['pm'].append(results.loc['zpm']['Z_Score'])
                model_results['phit'].append(results.loc['C(phit)[T.1]']['Z_Score'])
                model_results['phit:rpe'].append(results.loc['C(phit)[T.1]:zrpe']['Z_Score'])
                
                elec_counter +=1

    cue_model_df = pd.DataFrame(model_results)
    cue_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_rpe_pm_phit_intx_eleclevel_df', index=False)


In [ ]:
results

In [ ]:
band = 'beta'
analysis_evs = 'cue_on'
cue_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_rpe_pm_phit_intx_eleclevel_df')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_phit_rpe_eleclevel.pdf') as pdf:
    sns.barplot(data=cue_model_df, x='region', y='rpe',errorbar='se', 
            order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_df[np.abs(cue_model_df['rpe'])<=1.96], x='region', y='rpe', 
        order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=cue_model_df[np.abs(cue_model_df['rpe'])>1.96], x='region', y='rpe', 
        order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.5, color='red')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,4], ylim=[-3.5, 3.5])
    pdf.savefig(fig)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_phit_rpe_intx_eleclevel.pdf') as pdf:
    sns.barplot(data=cue_model_df, x='region', y='phit:rpe',errorbar='se', 
            order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_df[np.abs(cue_model_df['phit:rpe'])<=1.96], x='region', y='phit:rpe', 
        order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=cue_model_df[np.abs(cue_model_df['phit:rpe'])>1.96], x='region', y='phit:rpe', 
        order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.5, color='red')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,4], ylim=[-3.5, 3.5])
    pdf.savefig(fig)

In [ ]:
# Are the region-grouped beta values different from 0?
p_values = [] 
for region in ['dmPFC', 'HPC', 'OFC', 'AMY']: 
    p_values.append(ttest_1samp(cue_model_df[cue_model_df.region==region]['rpe'].values, popmean=0)[1])
    

[x*8 for x in p_values]

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(3, 5), dpi=300)
# sns.despine()
# with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_pm_eleclevel.pdf') as pdf:
#     sns.barplot(data=cue_model_df, x='region', y='pm',errorbar='se', 
#             order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha = 0.5, color='k')
#     sns.stripplot(data=cue_model_df, x='region', y='pm', 
#         order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.1, color='k')
#     sns.stripplot(data=cue_model_df[np.abs(cue_model_df.pm)>1.96], x='region', y='pm', 
#         order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.5, color='red')

#     plt.hlines([0], -1, 5, 'k')
#     ax.set(ylabel='PM sensitivity', xlim=[-1,4], ylim=[-3.5, 3.5])
#     pdf.savefig(fig)

In [ ]:
# # Are the region-grouped beta values different from 0?
# p_values = [] 
# for region in ['dmPFC', 'HPC', 'OFC', 'AMY']: 
#     p_values.append(ttest_1samp(cue_model_df[cue_model_df.region==region].pm.values, popmean=0)[1])
    

# [x*4 for x in p_values]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_phit_eleclevel.pdf') as pdf:
    sns.barplot(data=cue_model_df, x='region', y='phit',errorbar='se', 
            order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_df, x='region', y='phit', 
        order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=cue_model_df[np.abs(cue_model_df.phit)>1.96], x='region', y='phit', 
        order = ['dmPFC', 'HPC', 'OFC', 'AMY'], alpha=0.5, color='red')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='Memory sensitivity', xlim=[-1,4], ylim=[-3.5, 3.5])
    pdf.savefig(fig)

In [ ]:
# Are the region-grouped beta values different from 0?
p_values = [] 
for region in ['dmPFC', 'HPC', 'OFC', 'AMY']: 
    p_values.append(ttest_1samp(cue_model_df[cue_model_df.region==region].phit.values, popmean=0)[1])
    

[x*4 for x in p_values]

## single-region PAC

In [ ]:
roi = 'OFC'
metric = 'pac'
event = 'feedback_start' 
save_path = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'
all_elec_df = pd.read_csv(f'{save_path}/all_{roi}_pac_{event}.csv')

In [ ]:
all_elec_df.hemi = all_elec_df.hemi.str.lower()

In [ ]:
band = 'theta'
analysis_evs = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{analysis_evs}_elec_level_df_intx')
gamble_model_df = gamble_model_df[['participant', 'channel', 'region',  'rpe']]
gamble_model_df.rename(columns={'channel': 'label'}, inplace=True)

df1 = gamble_model_df.groupby(['participant', 'label', 'region']).mean(numeric_only=True).reset_index()

In [ ]:
test = df1.merge(df2, on=['participant', 'label', 'region'])

In [ ]:
sns.regplot(data=test[test.region=='OFC'], x='rpe', y='phit:zrpe')

Theta coding of RPE does not relate to the HFA coding of RPE:Phit interaction

In [ ]:
# OFC_coh_encoding = pd.read_csv('/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/coh/ACC_OFC_coh_theta_theta_feedback_start_df')
# OFC_coh_encoding['label'] = OFC_coh_encoding.pair_label.apply(lambda x: x.split(':')[1])


In [ ]:
df2 = all_elec_df.groupby(['participant', 'label']).mean(numeric_only=True).reset_index()

In [ ]:
sns.regplot(data=df1.merge(df2, on=['participant', 'label']), x='rpe_x', y='avg_conn')
plt.ylim([-0.15, .2])

No relationship between theta rpe enoding OR hfa rpe:encoding and PAC

In [ ]:
sns.regplot(x=all_elec_df.groupby(['participant', 'label']).mean(numeric_only=True).reset_index().avg_conn,
            y=half_quotient)

pac doesn't tell us much about whether an electrode will be an early or late responder

In [ ]:
band = 'hfa' 
event = 'feedback_start'
all_rpe_channels = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_time_res_regress_rec_split_OFC')
                        
          
hit_df = all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region']) & (all_rpe_channels['hit'] == True)]
miss_df = all_rpe_channels[(all_rpe_channels['index']=='rpe') & (all_rpe_channels['region'])& (all_rpe_channels['hit'] == False)]
contrast_df = hit_df.copy() 
contrast_df['Original_Estimate'] = hit_df.Original_Estimate.values - miss_df.Original_Estimate.values
first_half_df = contrast_df[contrast_df.ts<=750].groupby(['subj', 'label']).mean(numeric_only=True).reset_index()
second_half_df = contrast_df[contrast_df.ts>750].groupby(['subj', 'label']).mean(numeric_only=True).reset_index()
half_quotient = first_half_df.Original_Estimate - second_half_df.Original_Estimate

In [ ]:
# memory_df = combined_df[(combined_df.condition == 'Day1')].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
# memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
# all_elec_df = all_elec_df.merge(memory_df, on=['participant', 'trials'])

In [ ]:
sns.lmplot(data=all_elec_df, x='zrpe', y='avg_conn', hue='hemi', scatter=False)

In [ ]:
results = smf.mixedlm('avg_conn ~ 1 + zrpe*C(hemi) + zpm*C(hemi)', data=all_elec_df, 
                      re_formula = '1', 
                      groups='label').fit()

In [ ]:
results.summary()

In [ ]:
test_df = all_elec_df.groupby(['participant', 'label', 'hemi']).mean(numeric_only=True).reset_index()

In [ ]:
gamble_model_df.rename(columns={'channel': 'label'}, inplace=True)

In [ ]:
merge_df = gamble_model_df[gamble_model_df.region=='OFC'].merge(test_df, on =['participant', 'label']) 


In [ ]:
merge_df.rename(columns={'phit:zrpe': 'phit_rpe_intx'}, inplace=True)

In [ ]:
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/OFC_pac_phit:rpe.pdf') as pdf:

    sns.lmplot(data=merge_df, x='rpe', y='phit_rpe_intx', hue='participant', ci=None, 
               palette=sns.color_palette(['purple'], len(merged_df['participant'].unique())), 
               scatter_kws={'alpha':0.2}, line_kws={'alpha':0.2}, height=5, legend=None)
    sns.regplot(data=merge_df, x='avg_conn', y='phit_rpe_intx', color='k', ci=95, scatter=None)

    plt.ylim([-4, 4])
    
    pdf.savefig()

In [ ]:
results = smf.mixedlm('rpe_x ~ 1 + avg_conn*C(hemi)', data=merge_df, 
                      re_formula = '1', 
                      groups='participant').fit()

In [ ]:
results.summary()

In [ ]:
roi = 'OFC'
metric = 'pac'
event = 'feedback_start' 
save_path = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'
all_elec_df = pd.read_csv(f'{save_path}/all_{roi}_pac_{event}.csv')

all_elec_df['unique_elec_id'] = np.nan
counter = 0
for participant in all_elec_df.participant.unique():
    for label in all_elec_df[all_elec_df.participant==participant].label.unique():
        all_elec_df.loc[(all_elec_df.label==label) & (all_elec_df.participant==participant), 'unique_elec_id'] = counter
        counter+=1
        
all_elec_df.hemi = all_elec_df.hemi.str.lower()
    
# note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 


subj_list = all_elec_df[['participant', 'label']].drop_duplicates()['participant'].values
elec_list = all_elec_df[['participant', 'label']].drop_duplicates()['label'].values

model_results = {'participant':[], 'label':[], 'unique_elec_id':[],
                 'rpe':[], 'pm':[], 'phit':[]} 

for ix, _ in enumerate(elec_list):
    conn_df = all_elec_df[(all_elec_df.participant==subj_list[ix]) & (all_elec_df.label==elec_list[ix])]

    # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 

    if len(conn_df)>80:
        conn_df = conn_df.iloc[0:80]

    model_results['participant'].append(conn_df.participant.unique()[0])
    model_results['label'].append(conn_df.label.unique()[0])
    model_results['unique_elec_id'].append(conn_df.unique_elec_id.unique()[0])

    formula = 'sum_conn ~ 1 + zrpe + zpm'
    results = statistics_utils.permutation_regression_zscore(conn_df, 
                                            formula, 
                                            n_permutations=1000, 
                                            plot_res=False)

    model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
    model_results['pm'].append(results.loc['zpm']['Z_Score'])

    memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_list[ix])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
    memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
    memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
    new_df = conn_df.merge(memory_df, on=['participant', 'trials'])

    formula = 'sum_conn ~ 1 + zrpe*phit'
    results = statistics_utils.permutation_regression_zscore(new_df, 
                                            formula, 
                                            n_permutations=1000, 
                                            plot_res=False)

    model_results['phit'].append(results.loc['zrpe:phit']['Z_Score'])


test_df = pd.DataFrame(model_results)
test_df['region'] = roi

In [ ]:
test_df

In [ ]:
test_df

In [ ]:
sns.lmplot(data=new_df, x='zrpe', y='avg_conn', hue='phit')

In [ ]:
sns.barplot(data=test_df, x='region', y='phit',errorbar='se', alpha = 0.5, color='k')
sns.stripplot(data=test_df[np.abs(test_df.phit)<1.96], x='region', y='phit', alpha=0.1, color='k')
sns.stripplot(data=test_df[np.abs(test_df.phit)>1.96], x='region', y='phit', alpha=0.5, color='red')


## Cross-region PAC

In [ ]:
source_region = 'dlPFC'
metric = 'pac'
event = 'feedback_start' 
savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

all_pairs_df = []
for target_region in ['OFC']:
    for subj in subj_ids: 
        
        try:
            subj_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/pac/{subj}_{source_region}-{target_region}_pac_{event}.csv')
            all_pairs_df.append(subj_pairs_df)
        except FileNotFoundError:
            print('no pac here')
            continue
        
    all_pairs_df = pd.concat(all_pairs_df)
    

In [ ]:
all_pairs_df['label'] = all_pairs_df.label.apply(lambda x: x.split(':')[1])


In [ ]:
# all_pairs_df[all_pairs_df.participant=='MS012']n
all_pairs_df

In [ ]:
memory_df = combined_df[(combined_df.condition == 'Day1')].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
all_pairs_df = all_pairs_df.merge(memory_df, on=['participant', 'trials'])


In [ ]:
sns.lmplot(data=all_pairs_df, x='zrpe', y='sum_conn', hue='hits', scatter=False)


In [ ]:
all_pairs_df['phit'] = all_pairs_df.hits.apply(lambda x: 0.5 if x==True else -0.5)

In [ ]:
results = smf.mixedlm('sum_conn ~ 1 + zpm + zrpe + phit + zrpe:phit', 
                      re_formula ='1', 
                      groups='label', 
                      data=all_pairs_df).fit()

In [ ]:
results.summary()

In [ ]:
all_pairs_df

In [ ]:
# all_pairs_df['label'] = np.nan
# all_pairs_df['label'] = all_pairs_df.label.apply(lambda x: f'{x.split('-')[2]}-{x.split('-')[3]}')

In [ ]:
test_df = all_pairs_df.groupby(['participant', 'label']).mean(numeric_only=True).reset_index()

In [ ]:
merge_df = gamble_model_df[gamble_model_df.region=='OFC'].merge(test_df, on =['participant', 'label']) 


In [ ]:
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/AMY_OFC_pac_phit:rpe.pdf') as pdf:

    sns.lmplot(data=merge_df, x='avg_conn', y='phit:zrpe', hue='participant', ci=None, 
               palette=sns.color_palette(['purple'], len(merged_df['participant'].unique())), 
               scatter_kws={'alpha':0.2}, line_kws={'alpha':0.2}, height=5, legend=None)
    sns.regplot(data=merge_df, x='avg_conn', y='phit:zrpe', color='k', ci=95, scatter=None)

    plt.ylim([-4, 4])
    
    pdf.savefig()

In [ ]:
merge_df

In [ ]:
source_region = 'AMY'
metric = 'pac'
event = 'feedback_start' 
savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

all_dfs = []
for target_region in ['OFC']:
    all_pairs_df = []
    for subj in subj_ids[2:]: 
        
        try:
            subj_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/pac/{subj}_{source_region}-{target_region}_pac_{event}.csv')
            all_pairs_df.append(subj_pairs_df)
        except FileNotFoundError:
            continue
        
    all_pairs_df = pd.concat(all_pairs_df)
    # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 
    subj_list = all_pairs_df[['participant', 'label']].drop_duplicates()['participant'].values
    pair_list = all_pairs_df[['participant', 'label']].drop_duplicates()['label'].values

    model_results = {'participant':[], 'pair_label':[], 'roi1':[], 'roi2':[], 'unique_pair_id':[], 
                     'rpe':[], 'pm':[], 'phit':[]} 
    pair_counter = 0
    for ix, _ in enumerate(pair_list):
        conn_df = all_pairs_df[(all_pairs_df.participant==subj_list[ix]) & (all_pairs_df.label==pair_list[ix])]

        # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 

        if len(conn_df)>80:
            conn_df = conn_df.iloc[0:80]

        conn_df['unique_pair_id'] = pair_counter
        conn_df['zrpe'] = (conn_df.rpe - np.nanmean(conn_df.rpe)) / (2*np.nanstd(conn_df.rpe))
        conn_df['urpe'] = np.abs(conn_df.rpe)
        conn_df['zrpe_unsigned'] = (conn_df.urpe - np.nanmean(conn_df.urpe)) / (2*np.nanstd(conn_df.urpe))
        conn_df['zpm'] = (conn_df.DPRIME - np.nanmean(conn_df.DPRIME)) / (2*np.nanstd(conn_df.DPRIME))

        model_results['participant'].append(conn_df.participant.unique()[0])
        model_results['pair_label'].append(conn_df.label.unique()[0])
        model_results['roi1'].append(source_region)
        model_results['roi2'].append(target_region)
        model_results['unique_pair_id'].append(conn_df.unique_pair_id.unique()[0])

        formula = 'sum_conn ~ 1 + zrpe + zpm'
        results = permutation_regression_zscore(conn_df, 
                                                formula, 
                                                n_permutations=1000, 
                                                plot_res=False)

        model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
        model_results['pm'].append(results.loc['zpm']['Z_Score'])

        memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_list[ix])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
        memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
        memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
        new_df = conn_df.merge(memory_df, on=['participant', 'trials'])

        formula = 'sum_conn ~ 1 + zrpe*phit'
        results = permutation_regression_zscore(new_df, 
                                                formula, 
                                                n_permutations=1000, 
                                                plot_res=False)

        model_results['phit'].append(results.loc['zrpe:phit']['Z_Score'])

        pair_counter += 1

    test_df = pd.DataFrame(model_results)
    test_df['paired_regions'] = f'{source_region}_{target_region}'
    all_dfs.append(test_df)

In [ ]:
test_df = all_dfs[0]


In [ ]:
sns.barplot(data=test_df, x='roi1', y='phit',errorbar='se', alpha = 0.5, color='k')
sns.stripplot(data=test_df[np.abs(test_df.phit)<1.96], x='roi1', y='phit', alpha=0.1, color='k')
sns.stripplot(data=test_df[np.abs(test_df.phit)>1.96], x='roi1', y='phit', alpha=0.5, color='red')


In [ ]:
ttest_1samp(test_df.pm, popmean=0)

## Model single-electrode pair HFA AMP-AMP coupling during feedback_start



In [ ]:
source_region = 'HPC'
metric = 'wpli'
band1 = 'theta'
band2 = 'theta'
event = 'feedback_start' 
savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

all_dfs = []
for target_region in ['OFC']:

    all_pairs_df = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df')

    # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 

    subj_list = all_pairs_df[['participant', 'pair_label']].drop_duplicates()['participant'].values
    pair_list = all_pairs_df[['participant', 'pair_label']].drop_duplicates()['pair_label'].values

    model_results = {'participant':[], 'pair_label':[], 'roi1':[], 'roi2':[], 'unique_pair_id':[], 
                     'rpe':[], 'pm':[], 'phit':[]} 
    pair_counter = 0
    for ix, _ in enumerate(pair_list):
        conn_df = all_pairs_df[(all_pairs_df.participant==subj_list[ix]) & (all_pairs_df.pair_label==pair_list[ix])]

        # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 

        if len(conn_df)>80:
            conn_df = conn_df.iloc[0:80]

        conn_df['unique_pair_id'] = pair_counter
        conn_df['zrpe'] = (conn_df.rpe - np.nanmean(conn_df.rpe)) / (2*np.nanstd(conn_df.rpe))
        conn_df['urpe'] = np.abs(conn_df.rpe)
        conn_df['zrpe_unsigned'] = (conn_df.urpe - np.nanmean(conn_df.urpe)) / (2*np.nanstd(conn_df.urpe))
        conn_df['zpm'] = (conn_df.DPRIME - np.nanmean(conn_df.DPRIME)) / (2*np.nanstd(conn_df.DPRIME))

        model_results['participant'].append(conn_df.participant.unique()[0])
        model_results['pair_label'].append(conn_df.pair_label.unique()[0])
        model_results['roi1'].append(conn_df.roi1.unique()[0])
        model_results['roi2'].append(conn_df.roi2.unique()[0])
        model_results['unique_pair_id'].append(conn_df.unique_pair_id.unique()[0])

        formula = 'conn ~ 1 + zrpe + zpm'
        results = permutation_regression_zscore(conn_df, 
                                                formula, 
                                                n_permutations=1000, 
                                                plot_res=False)

        model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
        model_results['pm'].append(results.loc['zpm']['Z_Score'])

        memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_list[ix])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
        memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
        memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
        new_df = conn_df.merge(memory_df, on=['participant', 'trials'])

        formula = 'conn ~ 1 + zrpe*phit'
        results = permutation_regression_zscore(new_df, 
                                                formula, 
                                                n_permutations=1000, 
                                                plot_res=False)

        model_results['phit'].append(results.loc['zrpe:phit']['Z_Score'])

        pair_counter += 1

    test_df = pd.DataFrame(model_results)
    test_df['paired_regions'] = f'{source_region}_{target_region}'
    all_dfs.append(test_df)

In [ ]:
full_df = pd.concat(all_dfs)

In [ ]:
full_df

In [ ]:
# Save full df so I don't have to keep re-running it
full_df.to_csv(f'{savepath}/{metric}_conn_regress_results_{source_region}_{target_region}.csv', index=False)



In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(2, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_theta_{metric}_rpe_amp_{source_region}_{target_region}.pdf') as pdf:
    sns.barplot(data=full_df, x='paired_regions', y='rpe',errorbar='se', alpha = 0.5, color='k')
    sns.stripplot(data=full_df, x='paired_regions', y='rpe', alpha=0.1, color='k')
    sns.stripplot(data=full_df[np.abs(full_df.rpe)>1.96], x='paired_regions', y='rpe', alpha=0.5, color='red')

    plt.hlines([0], -1, 4, 'k')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,1], ylim=[-3.5, 3.5])
    pdf.savefig(fig)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(2, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_theta_{metric}_phit_amp_{source_region}_{target_region}.pdf') as pdf:
    sns.barplot(data=full_df, x='paired_regions', y='phit',errorbar='se', alpha = 0.5, color='k')
    sns.stripplot(data=full_df, x='paired_regions', y='phit', alpha=0.1, color='k')
    sns.stripplot(data=full_df[np.abs(full_df.phit)>1.96], x='paired_regions', y='phit', alpha=0.5, color='red')

    plt.hlines([0], -1, 4, 'k')
    ax.set(ylabel='Hit sensitivity', xlim=[-1,1], ylim=[-3.5, 3.5])
    pdf.savefig(fig)

## Model single-electrode pair coherence during feedback_start

In [ ]:
source_region = 'AMY'
metric = 'coh'
band1 = 'beta'
band2 = 'beta'
event = 'feedback_start' 
savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

all_dfs = []
for target_region in ['OFC', 'HPC']:

    all_pairs_df = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df')

    # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 


    subj_list = all_pairs_df[['participant', 'pair_label']].drop_duplicates()['participant'].values
    pair_list = all_pairs_df[['participant', 'pair_label']].drop_duplicates()['pair_label'].values

    model_results = {'participant':[], 'pair_label':[], 'roi1':[], 'roi2':[], 'unique_pair_id':[], 
                     'rpe':[], 'pm':[], 'phit':[], 'rpe:phit':[]} 
    pair_counter = 0
    for ix, _ in enumerate(pair_list):
        conn_df = all_pairs_df[(all_pairs_df.participant==subj_list[ix]) & (all_pairs_df.pair_label==pair_list[ix])]

        # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 

        if len(conn_df)>80:
            conn_df = conn_df.iloc[0:80]

        conn_df['unique_pair_id'] = pair_counter
        conn_df['zrpe'] = (conn_df.rpe - np.nanmean(conn_df.rpe)) / (2*np.nanstd(conn_df.rpe))
        conn_df['urpe'] = np.abs(conn_df.rpe)
        conn_df['zrpe_unsigned'] = (conn_df.urpe - np.nanmean(conn_df.urpe)) / (2*np.nanstd(conn_df.urpe))
        conn_df['zpm'] = (conn_df.DPRIME - np.nanmean(conn_df.DPRIME)) / (2*np.nanstd(conn_df.DPRIME))

        model_results['participant'].append(conn_df.participant.unique()[0])
        model_results['pair_label'].append(conn_df.pair_label.unique()[0])
        model_results['roi1'].append(conn_df.roi1.unique()[0])
        model_results['roi2'].append(conn_df.roi2.unique()[0])
        model_results['unique_pair_id'].append(conn_df.unique_pair_id.unique()[0])

        formula = 'conn ~ 1 + zrpe + zpm'
        results = permutation_regression_zscore(conn_df, 
                                                formula, 
                                                n_permutations=1000, 
                                                plot_res=False)

        model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
        model_results['pm'].append(results.loc['zpm']['Z_Score'])

        memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_list[ix])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
        memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
        memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
        new_df = conn_df.merge(memory_df, on=['participant', 'trials'])

        formula = 'conn ~ 1 + zrpe + phit + zrpe:phit'
        
        results = permutation_regression_zscore(new_df, 
                                                formula, 
                                                n_permutations=1000, 
                                                plot_res=False)

        model_results['phit'].append(results.loc['phit']['Z_Score'])
        model_results['rpe:phit'].append(results.loc['zrpe:phit']['Z_Score'])

        pair_counter += 1

    test_df = pd.DataFrame(model_results)
    test_df['paired_regions'] = f'{source_region}_{target_region}'
    all_dfs.append(test_df)

In [ ]:
full_df = pd.concat(all_dfs)

In [ ]:
full_df.paired_regions.unique()

In [ ]:
# Save full df so I don't have to keep re-running it
full_df.to_csv(f'{savepath}/beta_connectivity_regress_results_{source_region}_zrpe_phit.csv', index=False)



In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(2, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_beta_zrpe_phit_coh_{source_region}_{target_region}.pdf') as pdf:
    sns.barplot(data=full_df, x='paired_regions', y='phit',errorbar='se', alpha = 0.5, color='k')
    sns.stripplot(data=full_df, x='paired_regions', y='phit', alpha=0.1, color='k')
    sns.stripplot(data=full_df[np.abs(full_df['phit'])>1.96], x='paired_regions', y='phit', alpha=0.5, color='red')

    plt.hlines([0], -1, 4, 'k')
    ax.set(ylabel='RPE:PHIT sensitivity', xlim=[-1,2], ylim=[-3.5, 3.5])
    pdf.savefig(fig)
    

## Model single-electrode pair granger causality during feedback_start

In [ ]:
pairs = [['AMY', 'OFC'], ['AMY', 'HPC'], ['HPC', 'OFC']]
metric = 'granger'
band1 = 'theta'
band2 = 'theta'
event = 'feedback_start' 
savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

all_dfs = []
for pair in pairs:

    all_pairs_df = pd.read_csv(f'{savepath}/{pair[0]}_{pair[1]}_{metric}_{band1}_{band2}_{event}_df')

    # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 

    subj_list = all_pairs_df[['participant', 'pair_label']].drop_duplicates()['participant'].values
    pair_list = all_pairs_df[['participant', 'pair_label']].drop_duplicates()['pair_label'].values

    model_results = {'participant':[], 'pair_label':[], 'roi1':[], 'roi2':[], 'unique_pair_id':[], 
                     'rpe':[], 'pm':[], 'phit':[], 'rpe:phit':[]} 
    pair_counter = 0
    for ix, _ in enumerate(pair_list):
        conn_df = all_pairs_df[(all_pairs_df.participant==subj_list[ix]) & (all_pairs_df.pair_label==pair_list[ix])]

        # note the duplicates (all from MS028 for some reason.....) make a mismatch between the total size and elec pair length 

        if len(conn_df)>80:
            conn_df = conn_df.iloc[0:80]

        conn_df['unique_pair_id'] = pair_counter
        conn_df['zrpe'] = (conn_df.rpe - np.nanmean(conn_df.rpe)) / (2*np.nanstd(conn_df.rpe))
        conn_df['urpe'] = np.abs(conn_df.rpe)
        conn_df['zrpe_unsigned'] = (conn_df.urpe - np.nanmean(conn_df.urpe)) / (2*np.nanstd(conn_df.urpe))
        conn_df['zpm'] = (conn_df.DPRIME - np.nanmean(conn_df.DPRIME)) / (2*np.nanstd(conn_df.DPRIME))

        model_results['participant'].append(conn_df.participant.unique()[0])
        model_results['pair_label'].append(conn_df.pair_label.unique()[0])
        model_results['roi1'].append(conn_df.roi1.unique()[0])
        model_results['roi2'].append(conn_df.roi2.unique()[0])
        model_results['unique_pair_id'].append(conn_df.unique_pair_id.unique()[0])

        formula = 'conn ~ 1 + zrpe + zpm'
        results = permutation_regression_zscore(conn_df, 
                                                formula, 
                                                n_permutations=1000, 
                                                plot_res=False)

        model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
        model_results['pm'].append(results.loc['zpm']['Z_Score'])

        memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==subj_list[ix])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
        memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
        memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
        new_df = conn_df.merge(memory_df, on=['participant', 'trials'])

        formula = 'conn ~ 1 + zrpe*phit'
        
        results = permutation_regression_zscore(new_df, 
                                                formula, 
                                                n_permutations=1000, 
                                                plot_res=False)

        model_results['phit'].append(results.loc['phit']['Z_Score'])
        model_results['rpe:phit'].append(results.loc['zrpe:phit']['Z_Score'])

        pair_counter += 1

    test_df = pd.DataFrame(model_results)
    test_df['paired_regions'] = f'{pair[0]}_{pair[1]}'
    all_dfs.append(test_df)

In [ ]:
full_df = pd.concat(all_dfs)

In [ ]:
full_df.paired_regions.unique()

In [ ]:
# Save full df so I don't have to keep re-running it
full_df.to_csv(f'{savepath}/theta_connectivity_regress_results_zrpe_phit.csv', index=False)



In [ ]:
full_df = pd.read_csv(f'{savepath}/theta_connectivity_regress_results_zrpe_phit.csv')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(2, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_theta_zrpe_phit_gr_tc.pdf') as pdf:
    sns.barplot(data=full_df, x='paired_regions', y='rpe:phit',errorbar='se', alpha = 0.5, color='k')
    sns.stripplot(data=full_df[np.abs(full_df['rpe:phit'])<=1.96], x='paired_regions', y='rpe:phit', alpha=0.1, color='k')
    sns.stripplot(data=full_df[np.abs(full_df['rpe:phit'])>1.96], x='paired_regions', y='rpe:phit', alpha=0.5, color='red')

    plt.hlines([0], -1, 4, 'k')
    ax.set(ylabel='RPE:PHIT sensitivity')
    pdf.savefig(fig)
    

In [ ]:
# Are the region-grouped beta values different from 0?
p_values = [] 
for region in full_df.paired_regions.unique(): 
    p_values.append(ttest_1samp(full_df[full_df.paired_regions==region].rpe.values, popmean=0)[1])
    
    

In [ ]:
[x*2 for x in p_values]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_theta_zrpe_gr_tc.pdf') as pdf:
    sns.barplot(data=full_df, x='paired_regions', y='rpe',errorbar='se', alpha = 0.5, color='k')
    sns.stripplot(data=full_df[np.abs(full_df['rpe'])<=1.96], x='paired_regions', y='rpe', alpha=0.1, color='k')
    sns.stripplot(data=full_df[np.abs(full_df['rpe'])>1.96], x='paired_regions', y='rpe', alpha=0.5, color='red')

    plt.hlines([0], -1, 3, 'k')
    ax.set(ylabel='RPE sensitivity', ylim = [-3.75, 3.75])
    pdf.savefig(fig)
    

In [ ]:
# Are the region-grouped beta values different from 0?
p_values = [] 
for region in full_df.paired_regions.unique(): 
    p_values.append(ttest_1samp(full_df[full_df.paired_regions==region].phit.values, popmean=0)[1])
    
    

In [ ]:
region

In [ ]:
[x*4 for x in p_values]

In [ ]:
ttest_1samp(gamble_model_df_theta_coh.rpe, popmean=0)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 4), dpi=300, sharex=True, sharey=True)
# sns.regplot(data=full_df[full_df.paired_regions=='AMY_dmPFC'], x='rpe', y='phit', ax=ax[0])
# sns.regplot(data=full_df[full_df.paired_regions=='AMY_ACC'], x='rpe', y='phit', ax=ax[1])
sns.regplot(data=full_df[full_df.paired_regions=='AMY_OFC'], x='rpe', y='phit', ax=ax[2])
sns.regplot(data=full_df[full_df.paired_regions=='AMY_HPC'], x='rpe', y='phit', ax=ax[3])

sns.despine()
# pdf.savefig()

In [ ]:
# Are the region-grouped beta values different from 0?
p_values = [] 
spearman_r = []
for region in full_df.paired_regions.unique(): 
    p_values.append(spearmanr(full_df[full_df.paired_regions==region].rpe.values, 
                             full_df[full_df.paired_regions==region].phit.values,)[1])
    spearman_r.append(spearmanr(full_df[full_df.paired_regions==region].rpe.values, 
                             full_df[full_df.paired_regions==region].phit.values,)[0])

In [ ]:
p_values

## In addition, let's assess the relationship between the connectivity and individual electrode features: 

In [ ]:
full_df

In [ ]:
target_region = 'OFC'
all_pairs_df = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df')


In [ ]:
# Load the HFA dataframe 
band = 'hfa'
event = 'feedback_start'
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_rpe_pm_phit_eleclevel_df')


In [ ]:
# First, consolidate all electrode dfs, collapse the dumb manual columns into one column
all_elec_dfs = []
# subj_ids = ['UI001', 'UI002', 'UI003', 'UI004']
for subj_id in subj_ids: 
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
    elec_df['participant'] = subj_id
    all_elec_dfs.append(elec_df)
all_elec_dfs = pd.concat(all_elec_dfs).reset_index() 

In [ ]:
all_elec_dfs.rename(columns={'label':'channel'}, inplace=True)

In [ ]:


def split_electrode_pair(pair):
    return pair.split(':')

def add_electrode_features(df1, df2, feature):
    # Split the electrode pairs into two columns
    df2[['electrode1', 'electrode2']] = df2['pair_label'].apply(split_electrode_pair).tolist()
    
    # Create a multi-index dictionary for quick lookup of features from df1
    feature_dict = df1.set_index(['participant', 'channel'])[feature].to_dict()
    
    # Function to safely get feature value
    def get_feature(subject, electrode):
        return feature_dict.get((subject, electrode), None)
    
    # Add features for each electrode
    df2[f'electrode1_{feature}'] = df2.apply(lambda row: get_feature(row['participant'], row['electrode1']), axis=1)
    df2[f'electrode2_{feature}'] = df2.apply(lambda row: get_feature(row['participant'], row['electrode2']), axis=1)
    
    # Drop the temporary columns
    df2.drop(['electrode1', 'electrode2'], axis=1, inplace=True)
    
    return df2

# Example usage
df2 = add_electrode_features(gamble_model_df, full_df, 'rpe')
# df2 = add_electrode_features(gamble_model_df, df2, 'phit')
# df2 = add_electrode_features(all_elec_dfs, df2, 'mni_y')

# If you have multiple features, you can repeat the process
# df2 = add_electrode_features(df1, df2, 'another_feature')

In [ ]:
test_model = smf.ols(data=df2[df2.roi2=='HPC'], 
                     formula='rpe ~ 1 + electrode1_phit + electrode2_phit + electrode1_rpe + electrode2_rpe + electrode1_phit:electrode1_rpe').fit()


In [ ]:
test_model.summary()

In [ ]:
sns.regplot(y=test_model.resid, x=df2[df2.roi2=='HPC'].electrode1_phit)

In [ ]:
test_model.summary()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(3, 3), dpi=300)
sns.regplot(data=df2[df2.roi2=='HPC'], x='electrode1_phit', y='rpe', ax=ax, scatter=False, ci=95)
sns.despine()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(3, 3), dpi=300)
sns.regplot(data=df2[df2.roi2=='ACC'], x='electrode1_rpe', y='rpe', ax=ax, scatter=False)
sns.despine()

In [ ]:
merged_df = gamble_model_df.merge(all_elec_dfs, on=['participant', 'channel'])

sns.lmplot(data=merged_df, x='mni_z', y='rpe', hue='region', scatter=False)

In [ ]:
pearsonr(merged_df[merged_df.region=='ACC'].mni_z, merged_df[merged_df.region=='ACC'].rpe) 


In [ ]:
pearsonr(df2[df2.roi2=='HPC'].electrode1_phit, df2[df2.roi2=='HPC'].rpe)

The more an amygdala electrode vares about memory (HFA), the more input it receives from hippocampus as a function of RPE (theta)

## Model single-electrode responses during cue_on

In [ ]:
# MLE approach: skip this because it cannot use a t-distribution, and is thus not well correlated with Bayesian results
import patsy

rois = ['dmPFC', 'dlPFC', 'OFC', 'HPC', 'AMY']
block= [0, 0.5]
bands = ['theta', 'hfa']
analysis_evs = 'cue_on'
model_results = {'participant':[], 'channel':[], 'region':[], 'unique_elec_id':[], 
                 'rpe':[], 'pm':[], 'phit':[], 'phit:zrpe':[]} 

for band in bands: 
    
    # RELOAD the data each time the model is run to ensure no funny business with dataframe manipulation between loop runs
    chan_dfs = [] 
    glms = []
    elec_counter = 0 
    for subj_id in subj_ids:
        print(f'Loading subj {subj_id}')
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        for chan in elec_df.label.unique():
            if elec_df[elec_df.label==chan].salman_region.values[0] in rois:
                chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time{block[0]}-{block[1]}.csv')
                chan_df['unique_elec_id'] = elec_counter
                chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
                
                chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
                # Make a new response variable, memory outcome 
                chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome') 
                chan_df['phit'] = chan_df.apply(lambda x: 0.5 if x.memory_outcome=='hits' else -0.5 if x.memory_outcome=='misses' else np.nan, axis=1)

                model_results['participant'].append(chan_df.participant.unique()[0])
                model_results['channel'].append(chan_df.channel.unique()[0])
                model_results['region'].append(chan_df.region.unique()[0])
                model_results['unique_elec_id'].append(chan_df.unique_elec_id.unique()[0])
                
                formula = 'tfr ~ 1 + zrpe + zpm'
                results = statistics_utils.permutation_regression_zscore(chan_df[chan_df.fband==band], 
                                                        formula, 
                                                        n_permutations=1000, 
                                                        plot_res=False)
                
                model_results['rpe'].append(results.loc['zrpe']['Z_Score'])
                model_results['pm'].append(results.loc['zpm']['Z_Score'])
                
                formula = 'tfr ~ 1 + phit + phit*zrpe'
                results = statistics_utils.permutation_regression_zscore(chan_df[chan_df.fband==band], 
                                                        formula, 
                                                        n_permutations=1000, 
                                                        plot_res=False)
                
                model_results['phit'].append(results.loc['phit']['Z_Score'])
                model_results['phit:zrpe'].append(results.loc['phit:zrpe']['Z_Score'])
                elec_counter +=1

cue_model_df = pd.DataFrame(model_results)
cue_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_rpe_pm_phit_intx_eleclevel_df', index=False)


In [ ]:
band = 'hfa'
cue_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_rpe_pm_phit_intx_eleclevel_df')


In [ ]:
cue_model_df.region.unique()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_phit:zrpe_eleclevel.pdf') as pdf:
    sns.barplot(data=cue_model_df, x='region', y='phit:zrpe',errorbar='se', 
            order = ['dmPFC', 'dlPFC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_df, x='region', y='phit:zrpe', 
        order = ['dmPFC', 'dlPFC', 'OFC', 'HPC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=cue_model_df[np.abs(cue_model_df['phit:zrpe'])>1.96], x='region', y='phit:zrpe', 
        order = ['dmPFC', 'dlPFC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='red')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,5], ylim=[-3.5, 3.5])
    pdf.savefig(fig)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_rpe_eleclevel.pdf') as pdf:
    sns.barplot(data=cue_model_df, x='region', y='rpe',errorbar='se', 
            order = ['dmPFC', 'dlPFC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_df[np.abs(cue_model_df['rpe'])<=1.96], x='region', y='rpe', 
        order = ['dmPFC', 'dlPFC', 'OFC', 'HPC', 'AMY'], alpha=0.1, color='k')
    sns.stripplot(data=cue_model_df[np.abs(cue_model_df['rpe'])>1.96], x='region', y='rpe', 
        order = ['dmPFC', 'dlPFC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='red')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,5], ylim=[-3.5, 3.5])
    pdf.savefig(fig)

In [ ]:
region = 'AMY'
predictor = 'rpe'

if predictor == 'phit:zrpe':
    # rename this for our model
    cue_model_df.rename(columns={'phit:zrpe':'phit_rpe_intx'}, inplace=True)
    predictor = 'phit_rpe_intx'

model = bmb.Model(f'{predictor} ~ 1 + (1|participant)', 
                  cue_model_df[cue_model_df.region==region], 
                  family='gaussian', categorical=['participant'])

# Fit the model
results = model.fit(draws=5000, chains=4, target_accept=0.9)

df = az.summary(results)

# if results.pvalues['Intercept']<=0.025:
if np.sign(df.loc['Intercept']['hdi_3%']) == np.sign(df.loc['Intercept']['hdi_97%']):
    
    # print(results.summary())
    with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_{region}_{predictor}_M_on_betas.pdf') as pdf:

        # Create a scatter plot of individual data points
        fig, ax = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
        sns.stripplot(x="participant", y=f'{predictor}', data=cue_model_df[cue_model_df.region==region], 
                      jitter=True, alpha=0.6, color="gray", s=6, ax=ax)

        # plot the mean line
        subject_means = cue_model_df[cue_model_df.region==region].groupby("participant")[f'{predictor}'].mean()

        sns.boxplot(showmeans=True,
                    meanline=True,
                    meanprops={'color': 'k', 'ls': '-', 'lw': 2},
                    medianprops={'visible': False},
                    whiskerprops={'visible': False},
                    zorder=10,
                    x=subject_means.index,
                    y=subject_means.values,
                    showfliers=False,
                    showbox=False,
                    showcaps=False,
                    ax=ax)


        # # Overlay the subject means
        overall_mean = df.loc['Intercept']['mean']
        # Plot the overall mean from the mixed-effects model
        plt.axhline(overall_mean, color="blue", linestyle="--", label=f"Overall Mean: {overall_mean:.2f}")
        plt.axhline(df.loc['Intercept']['hdi_3%'], color='red', linestyle='--')
        plt.axhline(df.loc['Intercept']['hdi_97%'], color='red', linestyle='--')

        plt.axhline(0, color='black')
        sns.despine()
        
        pdf.savefig()

In [ ]:
df

In [ ]:
# Are the region-grouped beta values different from 0?
p_values = [] 
for region in rois: 
    p_values.append(ttest_1samp(cue_model_df[cue_model_df.region==region]['phit:zrpe'].values, popmean=0)[1])

In [ ]:
[x * 5 for x in p_values]

In [ ]:
palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300, sharex=True, sharey=True)
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_rpe_phit_corr.pdf') as pdf:

    sns.regplot(data=cue_model_df[cue_model_df.region=='dmPFC'], x='rpe', y='phit', color=dmpfc_c, ax=ax[0])
    sns.regplot(data=cue_model_df[cue_model_df.region=='ACC'], x='rpe', y='phit', color=acc_c, ax=ax[1])
    sns.regplot(data=cue_model_df[cue_model_df.region=='OFC'], x='rpe', y='phit', color=ofc_c, ax=ax[2])
    sns.regplot(data=cue_model_df[cue_model_df.region=='HPC'], x='rpe', y='phit', color=hpc_c, ax=ax[3])
    sns.regplot(data=cue_model_df[cue_model_df.region=='AMY'], x='rpe', y='phit', color=amy_c, ax=ax[4])
    sns.despine()
    pdf.savefig()
    

In [ ]:
# Now let's compare encoding RPE to cue RPE

palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300, sharex=True, sharey=True)
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_cue_{band}_rpe_rpe_corr.pdf') as pdf:

    sns.regplot(y=cue_model_df[cue_model_df.region=='dmPFC'].rpe, 
                x=gamble_model_df[gamble_model_df.region=='dmPFC'].rpe, color=dmpfc_c, ax=ax[0])
    sns.regplot(y=cue_model_df[cue_model_df.region=='ACC'].rpe, 
                x=gamble_model_df[gamble_model_df.region=='ACC'].rpe, color=acc_c, ax=ax[1])
    sns.regplot(y=cue_model_df[cue_model_df.region=='OFC'].rpe, 
                x=gamble_model_df[gamble_model_df.region=='OFC'].rpe, color=ofc_c, ax=ax[2])
    sns.regplot(y=cue_model_df[cue_model_df.region=='HPC'].rpe, 
                x=gamble_model_df[gamble_model_df.region=='HPC'].rpe, color=hpc_c, ax=ax[3])
    sns.regplot(y=cue_model_df[cue_model_df.region=='AMY'].rpe, 
                x=gamble_model_df[gamble_model_df.region=='AMY'].rpe, color=amy_c, ax=ax[4])
    sns.despine()
    pdf.savefig()
    

In [ ]:
# Now let's compare encoding HIT to cue HIT

palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300, sharex=True, sharey=True)
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_cue_{band}_phit_phit_corr.pdf') as pdf:

    sns.regplot(y=cue_model_df[cue_model_df.region=='dmPFC'].phit, 
                x=gamble_model_df[gamble_model_df.region=='dmPFC'].phit, color=dmpfc_c, ax=ax[0])
    sns.regplot(y=cue_model_df[cue_model_df.region=='ACC'].phit, 
                x=gamble_model_df[gamble_model_df.region=='ACC'].phit, color=acc_c, ax=ax[1])
    sns.regplot(y=cue_model_df[cue_model_df.region=='OFC'].phit, 
                x=gamble_model_df[gamble_model_df.region=='OFC'].phit, color=ofc_c, ax=ax[2])
    sns.regplot(y=cue_model_df[cue_model_df.region=='HPC'].phit, 
                x=gamble_model_df[gamble_model_df.region=='HPC'].phit, color=hpc_c, ax=ax[3])
    sns.regplot(y=cue_model_df[cue_model_df.region=='AMY'].phit, 
                x=gamble_model_df[gamble_model_df.region=='AMY'].phit, color=amy_c, ax=ax[4])
    sns.despine()
    pdf.savefig()
    

So it doesn't seem like anything notable here is really happening during the cue period wrt to RPE sensitivity OR reactivation of RPE- or MEM- related activity 